# 우체국 금융사기 피해사례 가설 분석
## 0. 데이터 준비

개별 보이스피싱 피해사례를 이용하여 피해자의 특성, 사기유형, 사칭기관과 피해금액의 관계를 확인하고 실제 피해사례의 특성을 설명합니다.

이 Notebook은 기존 `가설검정.ipynb`에서 우체국 관련 셀을 분리한 독립 실행용 Notebook입니다.

## 0-1. 환경 준비

Google Drive를 연결하고 데이터 처리에 필요한 `pandas`, 경로 처리를 위한 `Path`만 불러옵니다. 그래프를 만들지 않으므로 폰트 설치 코드는 포함하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

## 0-2. 데이터 경로 설정

프로젝트 루트는 `BASE_DIR` 한 곳에서만 관리합니다. Google Drive에 올린 폴더명이 `이종열`과 다르면 아래 한 줄만 수정하세요.

In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/이종열')
DATA_DIR = BASE_DIR / '데이터'
HYPOTHESIS_DIR = BASE_DIR / '가설'
REFERENCE_DIR = BASE_DIR / '분석기준'

file_paths = {
    '우체국 피해사례': DATA_DIR / 'df_postal.csv',
}

print('프로젝트 루트:', BASE_DIR)
for name, path in file_paths.items():
    print(f'{name}: {path.name} / 존재={path.exists()}')

missing_files = [str(path) for path in file_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('다음 파일을 찾을 수 없습니다:\n' + '\n'.join(missing_files))

## 0-3. CSV 불러오기

`df_postal.csv`는 UTF-8 BOM(`utf-8-sig`)으로 읽습니다. 인코딩을 명시하고, 읽기에 실패하면 오류가 그대로 드러나도록 합니다.

In [ ]:
df_postal_raw = pd.read_csv(file_paths['우체국 피해사례'], encoding='utf-8-sig')

raw_dataframes = {
    '우체국 피해사례': df_postal_raw,
}

for name, df in raw_dataframes.items():
    print(f'{name}: {df.shape}')

### 우체국 데이터의 역할

우체국 데이터는 개별 금융사기 피해사례입니다. 전화 기반 보이스피싱 사례에서 피해자 특성, 사기유형, 사칭기관과 피해금액의 관계를 탐색합니다. 투자사기는 현재 서비스 범위에서 제외합니다.

## 0-4. 원본 데이터 기본 구조 확인

값을 변경하기 전에 `head`, `shape`, 컬럼명, `info`, 기초 통계를 확인합니다. 반복 출력만 줄이기 위해 작은 확인 함수를 사용합니다.

In [ ]:
def show_basic_structure(name, df):
    print('\n' + '=' * 80)
    print(name)
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    display(df.head())
    print('\n[info]')
    df.info()
    print('\n[숫자형 기초 통계]')
    display(df.describe())
    print('\n[전체 컬럼 요약]')
    display(df.describe(include='all').transpose())

for name, df in raw_dataframes.items():
    show_basic_structure(name, df)

## 0-5. 결측치 확인

컬럼별 결측 개수와 결측률을 함께 확인합니다. 이 단계에서는 결측 행을 삭제하거나 채우지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    missing_report = pd.DataFrame({
        '결측 개수': df.isna().sum(),
        '결측률(%)': (df.isna().mean() * 100).round(2),
    })
    print('\n', name)
    display(missing_report)

## 0-6. 중복 확인

완전히 같은 행과 우체국의 `중복후보` 표시를 구분해 확인합니다. 서로 다른 사건이 같은 값을 가질 수 있으므로 원본에서는 자동 삭제하지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    duplicate_count = int(df.duplicated().sum())
    print(f'{name}: 완전중복 추가 행 {duplicate_count}건')
    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)].sort_values(df.columns.tolist()).head(50))

In [ ]:
print('우체국 중복후보 분포')
display(df_postal_raw['중복후보'].value_counts(dropna=False).rename_axis('중복후보').to_frame('건수'))

print('중복후보=True인 행')
display(df_postal_raw[df_postal_raw['중복후보'].eq(True)])

## 0-7. 범주형 값 확인

고유값 수가 작고 가설에 필요한 범주만 확인합니다. 긴 자유서술 컬럼 전체에 `value_counts()`를 적용하지 않습니다.

In [ ]:
postal_category_columns = [
    '연령대', '피해자 성별', '사기유형', '사칭기관',
    '피해구제 신청사유', '접근매체', '전화_보이스피싱', '중복후보',
]

for column in postal_category_columns:
    print(f'\n[{column}] 고유값 수: {df_postal_raw[column].nunique(dropna=False)}')
    display(df_postal_raw[column].value_counts(dropna=False).to_frame('건수'))

## 0-8. dtype 및 값 형식 확인

컬럼명·문자열 앞뒤 공백과 숫자형이어야 할 값의 변환 실패 여부를 확인합니다. 우체국 `피해액` 컬럼은 원 단위입니다.

In [ ]:
for name, df in raw_dataframes.items():
    column_space_count = sum(column != column.strip() for column in df.columns)
    string_space_rows = {}
    for column in df.select_dtypes(include=['object', 'string']).columns:
        values = df[column].dropna().astype(str)
        count = int(values.ne(values.str.strip()).sum())
        if count > 0:
            string_space_rows[column] = count
    print(f'{name}: 컬럼명 공백={column_space_count}, 문자열 값 공백={string_space_rows}')

In [ ]:
expected_numeric_columns = {
    '우체국 피해사례': ['연령대', '최초 접수년', '최초 접수월', '피해액', '자료기준일'],
}

for name, columns in expected_numeric_columns.items():
    df = raw_dataframes[name]
    print(f'\n[{name}]')
    for column in columns:
        converted = pd.to_numeric(df[column], errors='coerce')
        new_missing = int((converted.isna() & df[column].notna()).sum())
        print(f'{column}: 현재 dtype={df[column].dtype}, 숫자 변환 실패={new_missing}건')

In [ ]:
print('[피해금액 단위 및 범위 확인]')
print('우체국 피해액 단위: 원')
display(df_postal_raw['피해액'].describe().to_frame())
print('피해액 0원 이하:', int(df_postal_raw['피해액'].le(0).sum()), '건')

## 0-9. 최소 전처리

원본 DataFrame을 보호하기 위해 복사본에서만 작업합니다. 컬럼명과 문자열의 앞뒤 공백을 안전하게 제거하고, 숫자형이어야 하는 컬럼만 명시적으로 변환합니다. 의미가 다른 범주를 임의로 합치거나 행을 삭제하지 않습니다.

In [ ]:
df_postal_clean = df_postal_raw.copy()

prepared_dataframes = {
    '우체국 피해사례 정리본': df_postal_clean,
}

for df in prepared_dataframes.values():
    df.columns = df.columns.str.strip()
    for column in df.select_dtypes(include=['object', 'string']).columns:
        df[column] = df[column].str.strip()

In [ ]:
postal_numeric = ['연령대', '최초 접수년', '최초 접수월', '피해액', '자료기준일']

for column in postal_numeric:
    df_postal_clean[column] = pd.to_numeric(df_postal_clean[column], errors='coerce')

print('숫자형 변환 후 새 결측 확인')
print('우체국 피해사례 정리본', int(df_postal_clean.isna().sum().sum()))

## 0-10. 우체국 파생변수 판단

`연령대`, `최초 접수년`, `최초 접수월`이 이미 존재하므로 재생성하지 않습니다. 고액피해 컬럼은 없으며, 현재 분석에서는 실제 피해금액 자체를 핵심 변수로 사용합니다. 임의의 기준으로 피해금액을 이진화하지 않고 고액피해 파생변수를 생성하지 않습니다.

In [ ]:
required_postal_columns = [
    '연령대', '피해자 성별', '최초 접수년', '최초 접수월', '피해액',
    '사기유형', '사칭기관', '전화_보이스피싱', '중복후보',
]
missing_required_columns = [
    column for column in required_postal_columns
    if column not in df_postal_clean.columns
]
high_loss_columns = [column for column in df_postal_clean.columns if '고액' in column]

print('필수 컬럼 누락:', missing_required_columns)
print('현재 고액피해 관련 컬럼:', high_loss_columns)
print('고액피해 변수 생성: 미생성(실제 피해금액 자체 분석을 우선)')

## 0-11. 우체국 분석대상 데이터 준비

서비스 범위에 맞게 `전화_보이스피싱=True`이면서 `사기유형!='투자사기'`인 사례만 선택하고, 이미 표시된 `중복후보=True`는 분석용 복사본에서 제외합니다. 원본 `df_postal_raw`과 정리본 `df_postal_clean`은 그대로 유지됩니다. 현재 투자사기 43건은 모두 `전화_보이스피싱=False`이지만, 서비스 범위를 코드에 명확히 남기기 위해 두 조건을 모두 사용합니다.

> `중복후보`는 사건 식별자가 없는 상태에서 확정 중복을 뜻하지 않을 수 있습니다. 따라서 1단계 전에 제외 기준이 적절한지 사람이 다시 확인해야 합니다.

In [ ]:
phone_mask = df_postal_clean['전화_보이스피싱'].eq(True)
investment_mask = df_postal_clean['사기유형'].eq('투자사기')
duplicate_candidate_mask = df_postal_clean['중복후보'].eq(True)

df_postal_analysis = df_postal_clean[
    phone_mask & ~investment_mask & ~duplicate_candidate_mask
].copy()

postal_filter_counts = {
    '원본 표본 수': len(df_postal_raw),
    '전화 보이스피싱': int(phone_mask.sum()),
    '전화 외 사례': int((~phone_mask).sum()),
    '투자사기': int(investment_mask.sum()),
    '전체 중복후보': int(duplicate_candidate_mask.sum()),
    '전화 사례 중 중복후보 제외': int((phone_mask & duplicate_candidate_mask).sum()),
    '최종 분석 표본': len(df_postal_analysis),
}

for label, count in postal_filter_counts.items():
    print(f'{label}: {count}건')

In [ ]:
analysis_category_columns = ['연령대', '피해자 성별', '사기유형', '사칭기관']
for column in analysis_category_columns:
    print(f'\n[최종 분석 표본 - {column}]')
    display(df_postal_analysis[column].value_counts(dropna=False).to_frame('건수'))

## 0-12. 전처리 결과 최종 확인

전처리 전후 shape, 결측, 완전중복, dtype과 생성 변수를 비교합니다. 큰 피해액은 핵심 분석 대상이므로 IQR 밖의 값도 삭제하지 않습니다.

In [ ]:
final_dataframes = {
    '우체국 분석용': df_postal_analysis,
}

comparison_rows = []
raw_for_comparison = {
    '우체국 분석용': df_postal_raw,
}

for name, final_df in final_dataframes.items():
    raw_df = raw_for_comparison[name]
    comparison_rows.append({
        '데이터': name,
        '전처리 전 shape': str(raw_df.shape),
        '전처리 후 shape': str(final_df.shape),
        '최종 결측': int(final_df.isna().sum().sum()),
        '최종 완전중복 추가 행': int(final_df.duplicated().sum()),
        '원본에서 삭제한 행': 0,
    })

display(pd.DataFrame(comparison_rows))

In [ ]:
print('[최종 dtype]')
for name, df in final_dataframes.items():
    print(f'\n{name}')
    print(df.dtypes.to_string())

print('\n[최종 생성 변수]')
print('우체국: 생성 없음(실제 피해금액 자체 분석을 우선)')
print('범주 통일: 실제 표기 차이가 확인되지 않아 적용하지 않음')
print('이상치 삭제: 0건')
print('원본 CSV 저장/덮어쓰기: 수행하지 않음')

## 0-13. 0단계 요약

아래 셀은 우체국 분석 표본의 실제 숫자를 계산해 요약합니다. 고액피해 여부는 별도 핵심 Target으로 만들지 않고 실제 피해금액 자체를 사용합니다.

In [ ]:
print('=' * 60)
print('0단계 우체국 데이터 준비 결과')
print('=' * 60)
print('\n[우체국 피해사례]')
for label, count in postal_filter_counts.items():
    print(f'{label}: {count}건')
print('최종 결측:', int(df_postal_analysis.isna().sum().sum()), '개')
print('최종 완전중복 추가 행:', int(df_postal_analysis.duplicated().sum()), '건')
print('고액피해 변수: 생성하지 않음(실제 피해금액 자체 분석을 우선)')
print('처리한 내용: 전화 기반 사례 선택, 투자사기와 중복후보를 분석용 복사본에서 제외')
print('원본에서 삭제한 행: 0건')

print('\n[사람이 확인할 사항]')
print('1. 중복후보 37건 중 전화 사례 35건을 제외하는 기준의 적절성')
print('2. U3/U7/U8은 현재 핵심 분석에서 제외하고 실제 피해금액 관계를 우선 사용')
print('3. 표본 수가 매우 작은 사기유형·사칭기관 범주의 향후 처리 방법')

print('=' * 60)
print('0단계 완료')
print('다음 단계: 1. 기본 EDA')
print('=' * 60)

## **0단계 우체국 데이터 준비 완료**

우체국 데이터의 로드, 구조 확인, 결측·중복·범주·dtype 확인과 전화 기반 분석 표본 준비를 완료했습니다.

# 1. 우체국 기본 EDA

0단계에서 준비한 `df_postal_analysis`를 사용해 피해자 특성, 피해금액, 사기유형과 사칭기관의 기본 현황을 확인합니다.

## 1-1. 시각화 환경과 한글 폰트 설정

Google Colab에 나눔고딕을 한 번만 설치한 뒤 현재 런타임의 Matplotlib에 직접 등록합니다. 이 방식은 일반적으로 런타임 재시작이 필요 없습니다. 설치 셀 실행 후에도 한글이 깨지면 셀을 한 번 더 실행하세요.

In [ ]:
!apt-get update -qq
!apt-get install -qq fonts-nanum

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('Matplotlib 한글 폰트:', plt.rcParams['font.family'])

## 1-2. 우체국 분석 표본과 피해자 특성

0단계에서 전화 기반 보이스피싱과 중복후보 제외 조건으로 만든 `df_postal_analysis`만 사용합니다. 각 범주의 건수와 전체 분석 표본에서 차지하는 비율을 함께 확인합니다.

In [ ]:
def make_frequency_table(series, sort_index=False):
    """범주별 건수와 비율을 같은 표에서 확인하기 위한 작은 함수입니다."""
    counts = series.value_counts(dropna=False, sort=not sort_index)
    if sort_index:
        counts = counts.sort_index()
    table = counts.rename('건수').to_frame()
    table['비율(%)'] = (table['건수'] / table['건수'].sum() * 100).round(2)
    return table

postal_age_table = make_frequency_table(df_postal_analysis['연령대'], sort_index=True)
postal_gender_table = make_frequency_table(df_postal_analysis['피해자 성별'])
postal_fraud_type_table = make_frequency_table(df_postal_analysis['사기유형'])
postal_impersonation_table = make_frequency_table(df_postal_analysis['사칭기관'])

print('최종 분석 표본 수:', len(df_postal_analysis), '건')
print('\n[연령대]')
display(postal_age_table)
print('[성별]')
display(postal_gender_table)
print('[사기유형]')
display(postal_fraud_type_table)
print('[사칭기관]')
display(postal_impersonation_table)

In [ ]:
display(postal_age_table)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(postal_age_table.index.astype(str), postal_age_table['건수'])
ax.set_title('우체국 분석 표본의 연령대별 피해자 수')
ax.set_xlabel('연령대')
ax.set_ylabel('피해자수(명)')
plt.tight_layout()
plt.show()

In [ ]:
display(postal_gender_table)

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(postal_gender_table.index.astype(str), postal_gender_table['건수'])
ax.set_title('우체국 분석 표본의 성별 피해자 수')
ax.set_xlabel('성별')
ax.set_ylabel('피해자수(명)')
plt.tight_layout()
plt.show()

## 1-3. 우체국 피해금액 분포

피해금액은 연속형 숫자 데이터이므로 범주별 막대그래프가 아니라 **금액 구간별 사례 수를 나타내는 히스토그램**으로 분포를 확인합니다. 표본 수, 평균, 중앙값, 최소·최대, 표준편차, Q1, Q3, IQR, IQR 이상치 상한과 이상치 후보 개수도 함께 확인합니다.

우체국 `피해액` 원본은 원 단위이며 시각화에서만 `피해액(원) / 10,000`으로 계산한 만원 단위를 사용합니다. 경찰청 피해금액의 억원 단위와 통합하거나 두 출처의 절대금액을 직접 비교하지 않습니다.

> 피해금액은 극단적으로 큰 값이 실제 고액 보이스피싱 피해일 수 있습니다. IQR 기준을 벗어난 값은 데이터 오류나 제거 대상으로 단정하지 않고, 실제 피해금액 분포를 해석할 때 함께 확인합니다.

In [ ]:
loss_describe = df_postal_analysis['피해액'].describe()
loss_q1 = loss_describe['25%']
loss_q3 = loss_describe['75%']
loss_iqr = loss_q3 - loss_q1
loss_lower_fence = loss_q1 - 1.5 * loss_iqr
loss_upper_fence = loss_q3 + 1.5 * loss_iqr
loss_skewness = df_postal_analysis['피해액'].skew()
loss_outlier_mask = (
    df_postal_analysis['피해액'].lt(loss_lower_fence)
    | df_postal_analysis['피해액'].gt(loss_upper_fence)
)
loss_outlier_count = int(loss_outlier_mask.sum())

postal_loss_summary = pd.DataFrame({
    '항목': [
        '표본 수', '평균', '중앙값', '최소값', '최대값', '표준편차',
        'Q1', 'Q3', 'IQR', 'IQR 이상치 상한', 'IQR 기준 이상치 개수', '왜도',
    ],
    '값': [
        loss_describe['count'], loss_describe['mean'], df_postal_analysis['피해액'].median(),
        loss_describe['min'], loss_describe['max'], loss_describe['std'],
        loss_q1, loss_q3, loss_iqr, loss_upper_fence, loss_outlier_count, loss_skewness,
    ],
    '단위': ['건', '원', '원', '원', '원', '원', '원', '원', '원', '원', '건', '없음'],
})

display(postal_loss_summary.style.format({'값': '{:,.2f}'}))
print(f"평균 피해액: {df_postal_analysis['피해액'].mean():,.0f}원")
print(f"중앙 피해액: {df_postal_analysis['피해액'].median():,.0f}원")
print(f"IQR 이상치 상한: {loss_upper_fence:,.0f}원")
print(f"IQR 기준 이상치 후보: {loss_outlier_count:,}건")

loss_outlier_amount_table = (
    df_postal_analysis.loc[loss_outlier_mask, '피해액']
    .value_counts()
    .sort_index()
    .rename_axis('피해액(원)')
    .to_frame('건수')
)
display(loss_outlier_amount_table)

In [ ]:
WON_PER_MANWON = 10_000
loss_amount_manwon = df_postal_analysis['피해액'].dropna() / WON_PER_MANWON

postal_loss_unit_check = pd.DataFrame({
    '피해액(원)': [10_000_000, 100_000_000],
    '피해금액(만원)': [10_000_000 / WON_PER_MANWON, 100_000_000 / WON_PER_MANWON],
})
display(postal_loss_unit_check)
assert postal_loss_unit_check['피해금액(만원)'].tolist() == [1_000, 10_000]

histogram_counts, histogram_edges = np.histogram(loss_amount_manwon, bins=15)
postal_loss_histogram_table = pd.DataFrame({
    '구간_시작(만원)': histogram_edges[:-1],
    '구간_끝(만원)': histogram_edges[1:],
    '건수': histogram_counts,
})
display(postal_loss_histogram_table)

# 연속형 금액을 구간화한 빈도표를 구간 폭 그대로 붙여 그리므로 일반 범주 막대그래프가 아니라 히스토그램입니다.
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    postal_loss_histogram_table['구간_시작(만원)'],
    postal_loss_histogram_table['건수'],
    width=postal_loss_histogram_table['구간_끝(만원)'] - postal_loss_histogram_table['구간_시작(만원)'],
    align='edge', edgecolor='black',
)
ax.set_title('우체국 분석 표본의 피해금액 히스토그램')
ax.set_xlabel('피해금액(만원)')
ax.set_ylabel('사례 수(건)')
plt.tight_layout()
plt.show()

### 로그 변환 보조 히스토그램

전체 히스토그램은 초고액 사례까지 포함해 원자료의 전체 범위를 보여주지만, 강한 우측 왜도 때문에 비교적 낮은 금액대가 왼쪽에 압축될 수 있습니다. 아래 그래프는 `log1p(피해금액(만원))`을 사용해 그 구간을 더 자세히 확인합니다.

> 로그 변환은 이상치나 고액 피해 사례를 삭제한 것이 아니라, 오른쪽으로 긴 피해금액 분포를 시각적으로 확인하기 쉽게 만든 보조 시각화입니다. 원래 금액의 해석은 위 전체 히스토그램과 기초통계표를 기준으로 합니다.

In [ ]:
log_loss_amount = np.log1p(loss_amount_manwon)
log_histogram_counts, log_histogram_edges = np.histogram(log_loss_amount, bins=15)
postal_loss_log_histogram_table = pd.DataFrame({
    '로그구간_시작': log_histogram_edges[:-1],
    '로그구간_끝': log_histogram_edges[1:],
    '건수': log_histogram_counts,
})
display(postal_loss_log_histogram_table)

# 로그 변환 값도 연속 구간별 빈도를 계산한 뒤 같은 방식으로 히스토그램을 그립니다.
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    postal_loss_log_histogram_table['로그구간_시작'],
    postal_loss_log_histogram_table['건수'],
    width=(
        postal_loss_log_histogram_table['로그구간_끝']
        - postal_loss_log_histogram_table['로그구간_시작']
    ),
    align='edge', edgecolor='black',
)
ax.set_title('우체국 피해금액 로그 변환 히스토그램(보조)')
ax.set_xlabel('log1p(피해금액(만원))')
ax.set_ylabel('사례 수(건)')
plt.tight_layout()
plt.show()

In [ ]:
postal_loss_boxplot_data = df_postal_analysis[['피해액']].dropna().copy()
postal_loss_boxplot_data['피해액_만원'] = postal_loss_boxplot_data['피해액'] / WON_PER_MANWON
display(postal_loss_boxplot_data['피해액_만원'].describe().to_frame())

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.boxplot(postal_loss_boxplot_data['피해액_만원'], vert=False)
ax.set_title('우체국 분석 표본의 피해금액 박스플롯')
ax.set_xlabel('피해금액(만원)')
ax.set_yticks([])
plt.tight_layout()
plt.show()

### 일반 피해금액 영역 확대 박스플롯

전체 박스플롯에서 상자가 왼쪽에 작게 보이고 오른쪽에 점들이 나타나는 것은 코드 오류가 아니라, Q3에서 1.5×IQR을 넘는 고액 사례가 존재하는 우측 왜도 분포의 정상적인 표현입니다. 점은 IQR 기준 이상치 **후보**이며 실제 피해사례를 삭제하지 않습니다. 아래 그래프는 같은 전체 데이터를 사용하고 X축 표시 범위만 IQR 상한까지 확대합니다.

In [ ]:
postal_loss_zoom_range = pd.DataFrame({
    '표시_시작(만원)': [0],
    '표시_끝_IQR상한(만원)': [loss_upper_fence / WON_PER_MANWON],
    '그래프에_사용한_전체표본수': [len(postal_loss_boxplot_data)],
})
display(postal_loss_zoom_range)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.boxplot(postal_loss_boxplot_data['피해액_만원'], vert=False)
ax.set_xlim(0, loss_upper_fence / WON_PER_MANWON)
ax.set_title('우체국 피해금액 박스플롯(IQR 상한까지 확대)')
ax.set_xlabel('피해금액(만원)')
ax.set_yticks([])
plt.tight_layout()
plt.show()

### 분포 해석 범위

현재 데이터에서는 평균 피해금액이 중앙값보다 크고, 최대값이 Q3와 IQR 상한보다 크게 나타납니다. 전체 히스토그램과 박스플롯에서 오른쪽 꼬리와 고액 점들이 확인된다면 피해금액 분포가 우측으로 치우쳐 있고 일부 고액 피해 사례가 평균에 영향을 주는 모습으로 해석할 수 있습니다.

이는 1단계 EDA의 분포 관찰이며 피해금액이 커진 원인, 집단 간 차이 또는 통계적 유의성을 뜻하지 않습니다. IQR 밖의 사례도 실제 고액 보이스피싱 피해일 수 있으므로 임의로 제거하지 않고 전체 피해금액 분석에 유지합니다.

## 1-4. 우체국 사기유형과 사칭기관

사기유형과 사칭기관의 건수·비율을 확인합니다. 표본 수가 적은 범주도 현재는 자동 통합하지 않고 그대로 표시합니다.

In [ ]:
display(postal_fraud_type_table)

fraud_type_plot_table = postal_fraud_type_table.sort_values('건수')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fraud_type_plot_table.index.astype(str), fraud_type_plot_table['건수'])
ax.set_title('우체국 분석 표본의 사기유형별 피해사례 수')
ax.set_xlabel('사례 수(건)')
ax.set_ylabel('사기유형')
plt.tight_layout()
plt.show()

In [ ]:
display(postal_impersonation_table)

impersonation_plot_table = postal_impersonation_table.sort_values('건수')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(impersonation_plot_table.index.astype(str), impersonation_plot_table['건수'])
ax.set_title('우체국 분석 표본의 사칭기관별 피해사례 수')
ax.set_xlabel('사례 수(건)')
ax.set_ylabel('사칭기관')
plt.tight_layout()
plt.show()

In [ ]:
small_fraud_types = postal_fraud_type_table[postal_fraud_type_table['건수'] <= 5]
small_impersonation_types = postal_impersonation_table[postal_impersonation_table['건수'] <= 5]

print('[표본 5건 이하 사기유형 - 통합하지 않고 기록만 함]')
display(small_fraud_types)
print('[표본 5건 이하 사칭기관 - 통합하지 않고 기록만 함]')
display(small_impersonation_types)

## 1-5. 고액피해 변수 확인

0단계에서는 고액피해 파생변수를 생성하지 않았습니다. 현재 분석은 실제 피해금액의 크기 정보를 유지하는 방향이므로 임의 기준을 적용하지 않으며, 1단계에서도 고액피해 건수·비율 그래프를 만들지 않습니다. 고액피해 여부는 현재 핵심 분석 Target에 포함하지 않습니다.

In [ ]:
high_loss_candidates = [
    column for column in df_postal_analysis.columns
    if '고액피해' in column
]

if high_loss_candidates:
    high_loss_column = high_loss_candidates[0]
    postal_high_loss_table = make_frequency_table(df_postal_analysis[high_loss_column])
    display(postal_high_loss_table)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(postal_high_loss_table.index.astype(str), postal_high_loss_table['건수'])
    ax.set_title('고액피해 여부별 사례 수')
    ax.set_xlabel('고액피해 여부')
    ax.set_ylabel('사례 수(건)')
    plt.tight_layout()
    plt.show()
else:
    postal_high_loss_table = None
    print('고액피해 컬럼을 생성하지 않아 별도 집계와 그래프를 진행하지 않습니다.')

## 1-6. 우체국 접수 시기 기본 빈도

현재 분석 표본은 2025년 7~12월에 한정되어 있습니다. 연도·월별 건수만 확인하며 짧은 기간의 변화에 의미를 과도하게 부여하지 않습니다. 본격적인 시간 변화 분석은 이후 단계에서 진행합니다.

In [ ]:
postal_year_table = make_frequency_table(df_postal_analysis['최초 접수년'], sort_index=True)
postal_month_table = make_frequency_table(df_postal_analysis['최초 접수월'], sort_index=True)

print('[최초 접수년]')
display(postal_year_table)
print('[최초 접수월]')
display(postal_month_table)

# 우체국 기본 EDA 요약

아래 셀은 실행된 우체국 집계표에서 실제 값을 계산해 요약합니다. 수치는 하드코딩하지 않으며 관찰된 현황만 기술합니다.

In [ ]:
top_postal_age = postal_age_table['건수'].idxmax()
top_postal_gender = postal_gender_table['건수'].idxmax()
top_fraud_type = postal_fraud_type_table['건수'].idxmax()
top_impersonation = postal_impersonation_table['건수'].idxmax()

print('=' * 65)
print('우체국 기본 EDA 요약')
print('=' * 65)
print(f"분석 표본 수: {len(df_postal_analysis):,}건")
print(f"가장 많은 연령대: {top_postal_age}대 ({int(postal_age_table.loc[top_postal_age, '건수'])}건)")
print(f"가장 많은 성별: {top_postal_gender} ({int(postal_gender_table.loc[top_postal_gender, '건수'])}건)")
print(f"평균 피해금액: {df_postal_analysis['피해액'].mean():,.0f}원")
print(f"중앙 피해금액: {df_postal_analysis['피해액'].median():,.0f}원")
print(f"최대 피해금액: {df_postal_analysis['피해액'].max():,.0f}원")
print(f"가장 많은 사기유형: {top_fraud_type} ({int(postal_fraud_type_table.loc[top_fraud_type, '건수'])}건)")
print(f"가장 많은 사칭기관: {top_impersonation} ({int(postal_impersonation_table.loc[top_impersonation, '건수'])}건)")
if postal_high_loss_table is None:
    print('고액피해 건수 및 비율: 별도 Target을 만들지 않아 집계하지 않음')
else:
    print('고액피해 건수 및 비율: 위 고액피해 집계표 확인')
print(f"표본 5건 이하 사기유형: {small_fraud_types.index.astype(str).tolist()}")
print(f"표본 5건 이하 사칭기관: {small_impersonation_types.index.astype(str).tolist()}")

print('\n[다음 단계에서 확인할 관계 후보 - 아직 결론이 아님]')
print('- 사기유형별 피해금액 차이 여부')
print('- 사칭기관별 피해금액 차이 여부')
print('- 연령대별 피해금액 차이 여부')
print('- 성별 피해금액 차이 여부')
print('- 연령대·사기유형·사칭기관과 실제 피해금액의 관계')

print('\n[사람이 확인할 사항]')
print('- 고액피해 여부는 현재 핵심 분석 Target에서 제외')
print('- 5건 이하 소수 범주의 이후 분석 처리 방법')
print('- 2025년 7~12월에 한정된 우체국 표본의 대표성')

## **우체국 기본 EDA 완료**

우체국 분석 표본의 빈도, 비율, 피해금액 기초통계와 기본 분포를 확인했습니다. 피해금액은 이진화하지 않고 실제 금액 자체를 핵심 분석 변수로 유지합니다.

# 2. 전체 관계 스크리닝

이번 단계에서는 0단계에서 확정한 `df_postal_analysis`를 그대로 사용하여 우체국 개별 피해사례의 변수 관계를 넓게 탐색합니다. 각 검정의 **raw p-value와 효과크기**를 하나의 결과표에 모으되, 아직 다중검정 보정이나 최종 유의성 판정은 하지 않습니다.

경찰청 자료는 연도·지역·연령대별 집계자료이므로 개인 단위 관측치처럼 취급하지 않고, 이번 단계에서는 새로운 통계검정을 적용하지 않습니다.

## 2-1. 분석 기준과 공통 설정

1단계에서 피해금액의 강한 우측 왜도와 고액 사례가 확인되었으므로 정규성을 강하게 가정하지 않는 비모수 검정을 우선합니다.

- 3개 이상 그룹과 피해금액: Kruskal-Wallis, epsilon-squared
- 2개 그룹과 피해금액: Mann-Whitney U, rank-biserial correlation
- 범주형 변수끼리: 카이제곱 독립성 검정, Cramér's V

`ALPHA=0.05`는 raw p-value의 1차 후보 표시만 위한 기준입니다. 이 단계의 후보는 최종 유의 관계가 아닙니다.

In [ ]:
from scipy import __version__ as scipy_version
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu

ALPHA = 0.05
COMMON_INDEPENDENCE_NOTE = (
    '사건 간 독립성 가정; 0단계 중복후보 제외 기준의 타당성은 별도 확인 필요'
)
screening_rows = []

required_screening_columns = ['연령대', '피해자 성별', '피해액', '사기유형', '사칭기관']
missing_screening_columns = [
    column for column in required_screening_columns
    if column not in df_postal_analysis.columns
]
if missing_screening_columns:
    raise KeyError(f'2단계 필수 컬럼 누락: {missing_screening_columns}')

print('SciPy 버전:', scipy_version)
print('유의수준(ALPHA):', ALPHA)
print('분석 DataFrame: df_postal_analysis')
print('분석 표본 수:', len(df_postal_analysis))
print('필수 컬럼 누락:', missing_screening_columns)
print('고액피해 관련 컬럼:', [c for c in df_postal_analysis.columns if '고액피해' in c])

### 효과크기 해석 기준

효과크기는 지표별 참고 기준을 따로 사용합니다. 경계값은 절대적인 결론이 아니라 스크리닝을 위한 실무적 참고값입니다.

- epsilon-squared: 0.01 미만 매우 작음, 0.06 미만 작음, 0.14 미만 중간, 그 이상 큼
- rank-biserial correlation의 절대값: 0.10 미만 매우 작음, 0.30 미만 작음, 0.50 미만 중간, 그 이상 큼
- Cramér's V: 교차표의 작은 차원 `min(행-1, 열-1)`에 따라 Cohen 기준을 조정

카이제곱 검정은 일반적인 Cochran 기준인 `기대빈도 1 미만 없음`과 `기대빈도 5 미만 셀 20% 이하`를 진단합니다. 조건이 불안정해도 범주를 자동 통합하지 않고 raw p-value를 참고값으로 남깁니다.

In [ ]:
def make_amount_group_summary(data, group_column):
    """그룹별 피해액 표본 수와 기초통계를 검정 전에 확인합니다."""
    return (
        data.groupby(group_column, dropna=False)['피해액']
        .agg(표본수='count', 평균_피해액='mean', 중앙값_피해액='median', 최소값='min', 최대값='max')
        .sort_index()
    )

def epsilon_squared_kruskal(statistic, group_count, sample_count):
    """Kruskal-Wallis H 통계량에 대응하는 epsilon-squared를 계산합니다."""
    if sample_count <= group_count:
        return np.nan
    return max(0.0, (statistic - group_count + 1) / (sample_count - group_count))

def interpret_epsilon_squared(value):
    if pd.isna(value): return '계산 불가'
    if value < 0.01: return '매우 작음'
    if value < 0.06: return '작음'
    if value < 0.14: return '중간'
    return '큼'

def interpret_rank_biserial(value):
    magnitude = abs(value)
    if magnitude < 0.10: return '매우 작음'
    if magnitude < 0.30: return '작음'
    if magnitude < 0.50: return '중간'
    return '큼'

def cramers_v(chi2_statistic, sample_count, row_count, column_count):
    """카이제곱 통계량으로 Cramér's V를 계산합니다."""
    dimension = min(row_count - 1, column_count - 1)
    if sample_count == 0 or dimension <= 0:
        return np.nan
    return np.sqrt(chi2_statistic / (sample_count * dimension))

def interpret_cramers_v(value, row_count, column_count):
    """교차표 차원을 반영한 Cohen 참고 기준으로 Cramér's V를 해석합니다."""
    dimension = min(row_count - 1, column_count - 1)
    if pd.isna(value) or dimension <= 0:
        return '계산 불가'
    small = 0.10 / np.sqrt(dimension)
    medium = 0.30 / np.sqrt(dimension)
    large = 0.50 / np.sqrt(dimension)
    if value < small: return '매우 작음'
    if value < medium: return '작음'
    if value < large: return '중간'
    return '큼'

def first_candidate_label(raw_p_value):
    """raw p-value만으로 3단계 검토 대상을 표시하며 최종 판정은 하지 않습니다."""
    return '3단계 보정 검토' if raw_p_value < ALPHA else '현재 근거 약함'

def small_group_note(group_summary):
    small_groups = group_summary[group_summary['표본수'] < 5].index.astype(str).tolist()
    if small_groups:
        return f"소표본 그룹(<5): {small_groups}"
    return '표본 5건 미만 그룹 없음'

def add_small_effect_warning(note, raw_p_value, effect_interpretation):
    if raw_p_value < ALPHA and effect_interpretation in ['매우 작음', '작음']:
        return note + '; raw p는 작지만 효과크기가 작아 해석 주의'
    return note

In [ ]:
def run_chi_square_screening(test_id, relation, first_variable, second_variable, analysis_type):
    """교차표→카이제곱→기대빈도 진단→Cramér's V 순서로 실행하고 결과행을 추가합니다."""
    observed_table = pd.crosstab(
        df_postal_analysis[first_variable],
        df_postal_analysis[second_variable],
        dropna=False,
    )
    print(f'[{test_id}] 관측빈도: {relation}')
    display(observed_table)

    chi2_stat, raw_p, dof, expected = chi2_contingency(observed_table)
    expected_table = pd.DataFrame(expected, index=observed_table.index, columns=observed_table.columns)
    print('[기대빈도]')
    display(expected_table.style.format('{:.2f}'))

    expected_under_5 = int((expected < 5).sum())
    total_cells = int(expected.size)
    under_5_ratio = expected_under_5 / total_cells
    minimum_expected = float(expected.min())
    assumption_ok = minimum_expected >= 1 and under_5_ratio <= 0.20

    diagnostic_table = pd.DataFrame({
        '기대빈도_5미만_셀': [expected_under_5],
        '전체_셀': [total_cells],
        '5미만_비율(%)': [under_5_ratio * 100],
        '최소_기대빈도': [minimum_expected],
        '자유도': [dof],
        '가정충족여부': ['충족' if assumption_ok else '주의'],
    })
    display(diagnostic_table.style.format({'5미만_비율(%)': '{:.2f}', '최소_기대빈도': '{:.3f}'}))

    effect = cramers_v(chi2_stat, int(observed_table.to_numpy().sum()), *observed_table.shape)
    effect_interpretation = interpret_cramers_v(effect, *observed_table.shape)
    row_small = observed_table.sum(axis=1)[observed_table.sum(axis=1) < 5].index.astype(str).tolist()
    column_small = observed_table.sum(axis=0)[observed_table.sum(axis=0) < 5].index.astype(str).tolist()
    if row_small or column_small:
        sparse_note = f'소표본 범주(행): {row_small}; 소표본 범주(열): {column_small}'
    else:
        sparse_note = '관측 합계 5건 미만 범주 없음'
    expected_note = (
        f'기대빈도 5 미만 {expected_under_5}/{total_cells}셀({under_5_ratio:.1%}), '
        f'최소 기대빈도 {minimum_expected:.3f}'
    )
    if not assumption_ok:
        expected_note += '; 카이제곱 raw p-value는 참고값이며 해석 주의'
    note = add_small_effect_warning(
        expected_note + '; ' + sparse_note + '; ' + COMMON_INDEPENDENCE_NOTE,
        raw_p,
        effect_interpretation,
    )

    screening_rows.append({
        '검정ID': test_id,
        '관계': relation,
        '분석구분': analysis_type,
        '독립변수': first_variable,
        '종속변수': second_variable,
        '변수유형': '범주형 ↔ 범주형',
        '검정방법': '카이제곱 독립성 검정',
        '검정선택이유': '두 범주형 변수의 독립성 탐색; 기대빈도 진단을 함께 확인',
        '통계량': float(chi2_stat),
        '통계량_종류': 'chi-square',
        'raw_p_value': float(raw_p),
        '효과크기': float(effect),
        '효과크기_종류': "Cramér's V",
        '효과크기_해석': effect_interpretation,
        '표본수': int(observed_table.to_numpy().sum()),
        '그룹수': f'{observed_table.shape[0]}×{observed_table.shape[1]}',
        '가정충족여부': '충족' if assumption_ok else '주의',
        '주의사항': note,
        '1차후보': first_candidate_label(raw_p),
    })
    display(pd.DataFrame([screening_rows[-1]]))
    return observed_table, expected_table, diagnostic_table

## 2-2. U1 연령대 ↔ 피해금액

**연구 질문:** 연령대에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 연령대  
**종속변수:** 피해액

**분석 목적:** 연령대별 실제 피해금액 분포에 차이가 나타나는지 확인한다.

연령대는 3개 이상 그룹이고 피해금액은 우측 왜도와 고액 사례가 있으므로 Kruskal-Wallis 검정을 사용합니다. 유의하더라도 어느 그룹끼리 다른지는 현재 단계에서 확인하지 않습니다.

In [ ]:
age_amount_summary = make_amount_group_summary(df_postal_analysis, '연령대')
display(age_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
age_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('연령대', sort=True)
]
age_h, age_raw_p = kruskal(*age_groups)
age_effect = epsilon_squared_kruskal(age_h, len(age_groups), sum(len(group) for group in age_groups))
age_effect_interpretation = interpret_epsilon_squared(age_effect)
age_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(age_amount_summary)
    + '; 분포 모양이 다르면 중앙값 차이로만 해석하지 않음; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
age_note = add_small_effect_warning(age_note, age_raw_p, age_effect_interpretation)

screening_rows.append({
    '검정ID': 'S1', '관계': '연령대 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '연령대', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(age_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도와 고액 사례가 확인됨',
    'raw_p_value': float(age_raw_p), '효과크기': float(age_effect),
    '효과크기_종류': 'epsilon-squared', '효과크기_해석': age_effect_interpretation,
    '표본수': sum(len(group) for group in age_groups), '그룹수': len(age_groups),
    '가정충족여부': '대체로 충족', '주의사항': age_note,
    '1차후보': first_candidate_label(age_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-3. U2 성별 ↔ 피해금액

**연구 질문:** 성별에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 피해자 성별  
**종속변수:** 피해액

**분석 목적:** 성별에 따라 실제 피해금액 분포에 차이가 나타나는지 확인한다.

성별은 2개 그룹이며 피해금액의 왜도와 고액 사례가 커서 Mann-Whitney U 양측 검정을 사용합니다.

In [ ]:
gender_amount_summary = make_amount_group_summary(df_postal_analysis, '피해자 성별')
display(gender_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
gender_labels = sorted(df_postal_analysis['피해자 성별'].dropna().unique().tolist())
if len(gender_labels) != 2:
    raise ValueError(f'Mann-Whitney U 검정에는 2개 그룹이 필요합니다: {gender_labels}')
gender_group_1 = df_postal_analysis.loc[
    df_postal_analysis['피해자 성별'].eq(gender_labels[0]), '피해액'
].dropna().to_numpy()
gender_group_2 = df_postal_analysis.loc[
    df_postal_analysis['피해자 성별'].eq(gender_labels[1]), '피해액'
].dropna().to_numpy()
gender_u, gender_raw_p = mannwhitneyu(
    gender_group_1, gender_group_2, alternative='two-sided', method='asymptotic'
)
gender_effect = 2 * gender_u / (len(gender_group_1) * len(gender_group_2)) - 1
gender_effect_interpretation = interpret_rank_biserial(gender_effect)
gender_note = (
    f'그룹 순서: {gender_labels[0]} vs {gender_labels[1]}; 양수 효과크기는 첫 그룹의 순위가 더 큰 방향; '
    '동점이 존재할 수 있어 asymptotic 방식 사용; 분포 모양이 다르면 중앙값 차이로만 해석하지 않음; '
    + COMMON_INDEPENDENCE_NOTE
)
gender_note = add_small_effect_warning(gender_note, gender_raw_p, gender_effect_interpretation)

screening_rows.append({
    '검정ID': 'S2', '관계': '성별 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '피해자 성별', '종속변수': '피해액', '변수유형': '범주형(2그룹) ↔ 연속형',
    '검정방법': 'Mann-Whitney U', '통계량': float(gender_u), '통계량_종류': 'U',
    '검정선택이유': '2개 그룹이며 피해액의 우측 왜도와 고액 사례가 확인됨',
    'raw_p_value': float(gender_raw_p), '효과크기': float(gender_effect),
    '효과크기_종류': 'rank-biserial correlation',
    '효과크기_해석': gender_effect_interpretation,
    '표본수': len(gender_group_1) + len(gender_group_2), '그룹수': 2,
    '가정충족여부': '대체로 충족', '주의사항': gender_note,
    '1차후보': first_candidate_label(gender_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-4. U4 사기유형 ↔ 피해금액

**연구 질문:** 사기유형에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 사기유형  
**종속변수:** 피해액

**분석 목적:** 사기유형별 실제 피해금액 분포에 차이가 나타나는지 확인한다.

일부 유형의 표본이 매우 적으므로 범주를 삭제·통합하지 않고 Kruskal-Wallis 검정 결과에 소표본 주의를 기록합니다.

In [ ]:
fraud_amount_summary = make_amount_group_summary(df_postal_analysis, '사기유형')
display(fraud_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
fraud_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('사기유형', sort=True)
]
fraud_h, fraud_raw_p = kruskal(*fraud_groups)
fraud_effect = epsilon_squared_kruskal(fraud_h, len(fraud_groups), sum(len(group) for group in fraud_groups))
fraud_effect_interpretation = interpret_epsilon_squared(fraud_effect)
fraud_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(fraud_amount_summary)
    + '; 그룹별 표본 불균형이 커서 해석 주의; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
fraud_note = add_small_effect_warning(fraud_note, fraud_raw_p, fraud_effect_interpretation)

screening_rows.append({
    '검정ID': 'S3', '관계': '사기유형 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '사기유형', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(fraud_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도·고액 사례와 소표본 그룹이 확인됨',
    'raw_p_value': float(fraud_raw_p), '효과크기': float(fraud_effect),
    '효과크기_종류': 'epsilon-squared', '효과크기_해석': fraud_effect_interpretation,
    '표본수': sum(len(group) for group in fraud_groups), '그룹수': len(fraud_groups),
    '가정충족여부': '소표본 주의', '주의사항': fraud_note,
    '1차후보': first_candidate_label(fraud_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-5. U5 사칭기관 ↔ 피해금액

**연구 질문:** 사칭기관에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 사칭기관  
**종속변수:** 피해액

**분석 목적:** 사칭기관별 실제 피해금액 분포에 차이가 나타나는지 확인한다.

사칭기관은 3개 이상 그룹이며 소표본 기관이 포함되어 있어 Kruskal-Wallis 검정과 epsilon-squared를 계산하고 해석 주의를 기록합니다.

In [ ]:
impersonation_amount_summary = make_amount_group_summary(df_postal_analysis, '사칭기관')
display(impersonation_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
impersonation_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('사칭기관', sort=True)
]
impersonation_h, impersonation_raw_p = kruskal(*impersonation_groups)
impersonation_effect = epsilon_squared_kruskal(
    impersonation_h, len(impersonation_groups), sum(len(group) for group in impersonation_groups)
)
impersonation_effect_interpretation = interpret_epsilon_squared(impersonation_effect)
impersonation_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(impersonation_amount_summary)
    + '; 그룹별 표본 불균형이 커서 해석 주의; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
impersonation_note = add_small_effect_warning(
    impersonation_note, impersonation_raw_p, impersonation_effect_interpretation
)

screening_rows.append({
    '검정ID': 'S4', '관계': '사칭기관 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '사칭기관', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(impersonation_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도·고액 사례와 소표본 그룹이 확인됨',
    'raw_p_value': float(impersonation_raw_p), '효과크기': float(impersonation_effect),
    '효과크기_종류': 'epsilon-squared',
    '효과크기_해석': impersonation_effect_interpretation,
    '표본수': sum(len(group) for group in impersonation_groups),
    '그룹수': len(impersonation_groups), '가정충족여부': '소표본 주의',
    '주의사항': impersonation_note, '1차후보': first_candidate_label(impersonation_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-6. U6 사기유형 ↔ 사칭기관

**연구 질문:** 사기유형과 사칭기관 사이에 연관성이 있는가?

**변수:** 사기유형, 사칭기관

**분석 목적:** 반복되는 보이스피싱 시나리오 조합을 탐색한다.

두 변수 모두 범주형이므로 교차표를 먼저 확인한 뒤 카이제곱 독립성 검정과 Cramér's V를 계산합니다. 기대빈도 조건이 불안정하면 raw p-value는 참고값으로만 기록합니다.

In [ ]:
fraud_impersonation_observed, fraud_impersonation_expected, fraud_impersonation_diagnostic = (
    run_chi_square_screening(
        'S5', '사기유형 ↔ 사칭기관', '사기유형', '사칭기관', '핵심'
    )
)

## 2-7. 추가 피해자 ↔ 사기수법 스크리닝

다음 네 관계는 향후 피해자-사기수법 분석을 위한 탐색용 후보입니다. 이번 단계에서는 교차표, raw p-value, Cramér's V와 기대빈도 문제까지만 확인하며 취약집단 결론을 내리지 않습니다.

### S6. 연령대 ↔ 사기유형

In [ ]:
age_fraud_observed, age_fraud_expected, age_fraud_diagnostic = run_chi_square_screening(
    'S6', '연령대 ↔ 사기유형', '연령대', '사기유형', '추가 탐색'
)

### S7. 연령대 ↔ 사칭기관

In [ ]:
age_impersonation_observed, age_impersonation_expected, age_impersonation_diagnostic = (
    run_chi_square_screening(
        'S7', '연령대 ↔ 사칭기관', '연령대', '사칭기관', '추가 탐색'
    )
)

### S8. 성별 ↔ 사기유형

In [ ]:
gender_fraud_observed, gender_fraud_expected, gender_fraud_diagnostic = run_chi_square_screening(
    'S8', '성별 ↔ 사기유형', '피해자 성별', '사기유형', '추가 탐색'
)

### S9. 성별 ↔ 사칭기관

In [ ]:
gender_impersonation_observed, gender_impersonation_expected, gender_impersonation_diagnostic = (
    run_chi_square_screening(
        'S9', '성별 ↔ 사칭기관', '피해자 성별', '사칭기관', '추가 탐색'
    )
)

## 2-8. 현재 분석에서 진행하지 않는 가설

### U3. 연령대 ↔ 고액피해 여부
현재 분석에서는 추가 진행하지 않습니다. 연령대와 실제 피해금액의 관계를 우선 사용하며, 고액피해 기준을 임의로 설정하지 않습니다.

### U7. 사기유형 ↔ 고액피해 여부
현재 분석에서는 추가 진행하지 않으며, 사기유형과 실제 피해금액의 관계를 우선 사용합니다.

### U8. 사칭기관 ↔ 고액피해 여부
현재 분석에서는 추가 진행하지 않으며, 사칭기관과 실제 피해금액의 관계를 우선 사용합니다.

고액피해 여부를 분석하는 방법 자체가 잘못된 것은 아닙니다. 다만 현재 서비스 목적과 대시보드 분석 방향에서는 우선순위가 낮으므로 추가 핵심 분석에서 제외합니다.

In [ ]:
held_hypotheses = pd.DataFrame([
    {'가설ID': 'U3', '관계': '연령대 ↔ 고액피해 여부', '상태': '현재 분석 미진행', '사유': '연령대 ↔ 실제 피해금액 분석을 우선하며 고액피해 기준을 별도 설정하지 않음'},
    {'가설ID': 'U7', '관계': '사기유형 ↔ 고액피해 여부', '상태': '현재 분석 미진행', '사유': '사기유형 ↔ 실제 피해금액 분석을 우선하며 고액피해 기준을 별도 설정하지 않음'},
    {'가설ID': 'U8', '관계': '사칭기관 ↔ 고액피해 여부', '상태': '현재 분석 미진행', '사유': '사칭기관 ↔ 실제 피해금액 분석을 우선하며 고액피해 기준을 별도 설정하지 않음'},
])
display(held_hypotheses)

## 2-9. 전체 `screening_results`

실행한 9개 검정을 하나의 DataFrame으로 정리합니다. `raw_p_value`는 다음 단계에서 그대로 사용할 수 있도록 숫자형을 유지하고, 화면 표시용 p-value 컬럼만 별도로 만듭니다.

In [ ]:
screening_results = pd.DataFrame(screening_rows)

if screening_results['검정ID'].duplicated().any():
    duplicate_ids = screening_results.loc[
        screening_results['검정ID'].duplicated(keep=False), '검정ID'
    ].tolist()
    raise ValueError(f'중복 검정ID 발견: {duplicate_ids}')
if len(screening_results) != 9:
    raise ValueError(f'예상한 검정 수는 9개이지만 {len(screening_results)}개가 기록되었습니다.')

screening_results['raw_p_value'] = pd.to_numeric(screening_results['raw_p_value'])
screening_results['통계량'] = pd.to_numeric(screening_results['통계량'])
screening_results['효과크기'] = pd.to_numeric(screening_results['효과크기'])

def display_p_value(value):
    return f'{value:.3e}' if value < 0.001 else f'{value:.4f}'

screening_results_display = screening_results.copy()
screening_results_display['통계량'] = screening_results_display['통계량'].map(lambda value: f'{value:.4f}')
screening_results_display['raw_p_value_표시'] = screening_results_display['raw_p_value'].map(display_p_value)
screening_results_display['효과크기'] = screening_results_display['효과크기'].map(lambda value: f'{value:.4f}')

display_columns = [
    '검정ID', '관계', '분석구분', '검정방법', '검정선택이유', '통계량', 'raw_p_value_표시',
    '효과크기', '효과크기_종류', '효과크기_해석', '표본수',
    '가정충족여부', '주의사항', '1차후보',
]
display(screening_results_display[display_columns])
print('\n[원본 결과 dtype - raw_p_value는 숫자형 유지]')
print(screening_results.dtypes)

## 2-10. 2단계 결과 요약

아래 셀은 실행 결과에서 총 검정 수, raw p-value 후보, 효과크기, 가정 및 소표본 주의를 자동 집계합니다. 현재 결과는 다중검정 보정 전 스크리닝 결과이므로 최종 유의 관계로 확정하지 않습니다.

In [ ]:
raw_candidate_results = screening_results[screening_results['raw_p_value'] < ALPHA]
weak_raw_results = screening_results[screening_results['raw_p_value'] >= ALPHA]
medium_or_larger_results = screening_results[
    screening_results['효과크기_해석'].isin(['중간', '큼'])
]
chi_square_caution_results = screening_results[
    screening_results['검정방법'].eq('카이제곱 독립성 검정')
    & screening_results['가정충족여부'].eq('주의')
]
small_sample_caution_results = screening_results[
    screening_results['주의사항'].str.contains('소표본', na=False)
]
small_effect_raw_candidates = raw_candidate_results[
    raw_candidate_results['효과크기_해석'].isin(['매우 작음', '작음'])
]

screening_summary = pd.DataFrame({
    '항목': [
        '총 실행 검정 수', 'raw p < 0.05 관계 수', '효과크기 중간 이상 관계 수',
        '카이제곱 가정 주의 관계 수', '표본 부족 주의 관계 수',
        'raw p는 작지만 효과크기가 작은 관계 수',
    ],
    '값': [
        len(screening_results), len(raw_candidate_results), len(medium_or_larger_results),
        len(chi_square_caution_results), len(small_sample_caution_results),
        len(small_effect_raw_candidates),
    ],
})
display(screening_summary)

print('[3단계 보정 검토 후보 - 아직 최종 유의 관계가 아님]')
print(raw_candidate_results['관계'].tolist())
print('\n[현재 raw p-value 근거가 약한 관계]')
print(weak_raw_results['관계'].tolist())
print('\n[효과크기 중간 이상 관계]')
print(medium_or_larger_results[['관계', '효과크기', '효과크기_종류', '효과크기_해석']].to_dict('records'))
print('\n[카이제곱 가정 주의 관계]')
print(chi_square_caution_results['관계'].tolist())
print('\n[소표본 주의 관계]')
print(small_sample_caution_results['관계'].tolist())
print('\n[raw p는 작지만 효과크기가 작은 관계]')
print(small_effect_raw_candidates['관계'].tolist())
print('\n[현재 분석 미진행]')
for _, row in held_hypotheses.iterrows():
    print(f"- {row['가설ID']} {row['관계']}: {row['사유']}")

print('\n주의: 모든 p-value는 다중검정 보정 전 raw p-value입니다.')
print('3단계 보정 전에는 최종 유의 관계로 확정하지 않습니다.')

## **2단계 전체 관계 스크리닝 완료**

현재 Notebook에서는 우체국 개별 피해사례의 핵심 5개 관계와 추가 탐색 4개 관계에 대해 raw p-value, 효과크기와 가정 진단을 수집했습니다.

- 고액피해 관련 U3·U7·U8은 실제 피해금액 분석을 우선하여 현재 핵심 분석에서 제외했습니다.
- 경찰청 집계자료와 시간 변수에는 새로운 검정을 적용하지 않았습니다.
- 범주를 자동 삭제·통합하지 않았고 고액 피해 사례도 제거하지 않았습니다.
- 사후검정과 다중검정 보정은 수행하지 않았습니다.

Colab 실행 결과와 기대빈도·소표본 주의를 사람이 확인한 뒤 다음 요청에서 3단계 다중검정 보정을 진행합니다.

# 3. 다중검정 보정

## 3-1. 다중검정이 필요한 이유

2단계에서는 여러 관계를 동시에 검정했습니다. 검정 수가 많아질수록 실제 관계가 없어도 우연히 `p < 0.05`가 나올 가능성이 커집니다.

따라서 3단계에서는 S1~S9의 raw p-value 9개를 하나의 검정 family로 보고 Benjamini-Hochberg 방식으로 False Discovery Rate(FDR)를 통제합니다. 이는 모든 오류 가능성을 완전히 없애는 방법이 아니라, 유의하다고 판단한 결과 중 거짓 양성의 비율을 일정 수준으로 관리하는 방법입니다.

중요하게도 FDR 보정은 다중검정 문제만 다룹니다. 카이제곱 검정의 기대빈도 부족이나 소표본 범주 문제는 해결하지 않으므로, 보정 후에도 검정 가정 주의를 별도로 유지합니다.

## 3-2. 2단계 결과 검증

보정 전에 `screening_results`의 구조와 9개 raw p-value를 확인합니다. 조건이 맞지 않으면 계산을 중단해 잘못된 결과가 만들어지는 것을 막습니다.

In [ ]:
from pandas.api.types import is_numeric_dtype

if 'screening_results' not in globals():
    raise NameError('screening_results가 없습니다. 0~2단계 셀을 순서대로 먼저 실행해 주세요.')

required_fdr_columns = [
    '검정ID', '관계', '분석구분', '검정방법', '통계량', 'raw_p_value',
    '효과크기', '효과크기_종류', '효과크기_해석', '표본수',
    '가정충족여부', '주의사항', '1차후보',
]
missing_fdr_columns = [c for c in required_fdr_columns if c not in screening_results.columns]
if missing_fdr_columns:
    raise KeyError(f'3단계 필수 컬럼 누락: {missing_fdr_columns}')

expected_test_ids = [f'S{i}' for i in range(1, 10)]
actual_test_ids = screening_results['검정ID'].astype(str).tolist()
if len(screening_results) != 9:
    raise ValueError(f'FDR 보정 대상은 S1~S9의 9개 검정이어야 합니다: 현재 {len(screening_results)}개')
if screening_results['검정ID'].duplicated().any():
    duplicates = screening_results.loc[screening_results['검정ID'].duplicated(False), '검정ID'].tolist()
    raise ValueError(f'중복 검정ID 발견: {duplicates}')
if actual_test_ids != expected_test_ids:
    raise ValueError(f'검정ID와 순서가 S1~S9가 아닙니다: {actual_test_ids}')
if not is_numeric_dtype(screening_results['raw_p_value']):
    raise TypeError('raw_p_value는 숫자형이어야 합니다.')
if screening_results['raw_p_value'].isna().any():
    raise ValueError('raw_p_value에 결측치가 있습니다.')
if not screening_results['raw_p_value'].between(0, 1, inclusive='both').all():
    raise ValueError('raw_p_value는 모두 0 이상 1 이하여야 합니다.')

print('screening_results 검증 완료')
print('보정 대상 검정 수:', len(screening_results))
print('검정ID:', actual_test_ids)
print('raw_p_value dtype:', screening_results['raw_p_value'].dtype)

## 3-3. Benjamini-Hochberg 적용

raw 기준에서 유의한 결과만 고르지 않고, 계획한 S1~S9 전체를 한 번에 보정합니다. 2단계 결과를 보존하기 위해 별도 DataFrame인 `fdr_results`를 사용합니다.

In [ ]:
from statsmodels.stats.multitest import multipletests

original_columns = screening_results.columns.tolist()
original_test_ids = screening_results['검정ID'].tolist()
fdr_results = screening_results.copy()

reject, adjusted_p_values, _, _ = multipletests(
    fdr_results['raw_p_value'].to_numpy(),
    alpha=ALPHA,
    method='fdr_bh',
)
fdr_results['adjusted_p_value'] = adjusted_p_values.astype(float)
fdr_results['raw_유의여부'] = fdr_results['raw_p_value'] < ALPHA
fdr_results['FDR_통과여부'] = np.where(reject, '통과', '미통과')

assumption_caution = fdr_results['가정충족여부'].astype(str).str.contains('주의', na=False)
fdr_results['최종판정'] = np.select(
    [~reject, reject & assumption_caution],
    ['보정 후 근거 약함', 'FDR 통과 - 검정 가정 주의'],
    default='FDR 통과',
)

if fdr_results['adjusted_p_value'].isna().any():
    raise ValueError('adjusted_p_value에 결측치가 있습니다.')
if not fdr_results['adjusted_p_value'].between(0, 1, inclusive='both').all():
    raise ValueError('adjusted_p_value는 모두 0 이상 1 이하여야 합니다.')
if screening_results.columns.tolist() != original_columns:
    raise RuntimeError('원본 screening_results의 컬럼이 변경되었습니다.')
if fdr_results['검정ID'].tolist() != original_test_ids:
    raise RuntimeError('FDR 결과에서 S1~S9의 원래 순서가 변경되었습니다.')

print('Benjamini-Hochberg FDR 보정과 결과 검증 완료')

## 3-4. 보정 결과표

원본 숫자 컬럼은 float로 유지하고, 화면 표시용 컬럼만 별도로 만들어 매우 작은 p-value가 `0.000`으로 보이지 않게 합니다.

In [ ]:
fdr_display_columns = [
    '검정ID', '관계', '검정방법', 'raw_p_value', 'adjusted_p_value',
    '효과크기', '효과크기_종류', '효과크기_해석', '가정충족여부',
    'FDR_통과여부', '최종판정', '주의사항',
]
display(
    fdr_results[fdr_display_columns].style.format({
        'raw_p_value': display_p_value,
        'adjusted_p_value': display_p_value,
        '효과크기': '{:.4f}',
    })
)
print('raw_p_value dtype:', fdr_results['raw_p_value'].dtype)
print('adjusted_p_value dtype:', fdr_results['adjusted_p_value'].dtype)

## 3-5. raw p-value vs adjusted p-value 비교

In [ ]:
p_value_comparison = fdr_results[[
    '검정ID', '관계', 'raw_p_value', 'adjusted_p_value', 'raw_유의여부', 'FDR_통과여부'
]].copy()
p_value_comparison['raw_p_value_표시'] = p_value_comparison['raw_p_value'].map(display_p_value)
p_value_comparison['BH_adjusted_p_value_표시'] = p_value_comparison['adjusted_p_value'].map(display_p_value)
p_value_comparison['raw_기준_통과여부'] = np.where(p_value_comparison['raw_유의여부'], '통과', '미통과')
p_value_comparison['BH_기준_통과여부'] = p_value_comparison['FDR_통과여부']
display(p_value_comparison[[
    '검정ID', '관계', 'raw_p_value_표시', 'BH_adjusted_p_value_표시',
    'raw_기준_통과여부', 'BH_기준_통과여부',
]])

dropped_after_bh = fdr_results[
    fdr_results['raw_유의여부'] & fdr_results['FDR_통과여부'].eq('미통과')
]
print('[BH 보정 후 탈락 관계]')
print(dropped_after_bh['관계'].tolist())

## 3-6. 검정 가정을 포함한 판정

`adjusted p < 0.05`만으로 모든 관계를 같은 수준의 최종 유의 관계로 확정하지 않습니다. BH를 통과했더라도 2단계의 `가정충족여부`가 주의라면 별도 상태로 관리합니다.

In [ ]:
fdr_pass_results = fdr_results[fdr_results['FDR_통과여부'].eq('통과')]
fdr_good_assumption_results = fdr_results[fdr_results['최종판정'].eq('FDR 통과')]
fdr_caution_results = fdr_results[fdr_results['최종판정'].eq('FDR 통과 - 검정 가정 주의')]
fdr_weak_results = fdr_results[fdr_results['최종판정'].eq('보정 후 근거 약함')]
medium_or_larger_fdr_results = fdr_pass_results[
    fdr_pass_results['효과크기_해석'].isin(['중간', '큼'])
]

print('[raw p < 0.05 관계]')
print(fdr_results.loc[fdr_results['raw_유의여부'], '관계'].tolist())
print('\n[BH 보정 후 통과 관계]')
print(fdr_pass_results['관계'].tolist())
print('\n[BH 보정 후 탈락 관계]')
print(dropped_after_bh['관계'].tolist())
print('\n[BH 통과했지만 검정 가정 주의 관계]')
print(fdr_caution_results['관계'].tolist())
print('\n[BH 통과 + 효과크기 중간 이상 관계]')
print(medium_or_larger_fdr_results[['관계', '효과크기', '효과크기_종류', '효과크기_해석']].to_dict('records'))

## 3-7. 보정 전후 요약

In [ ]:
fdr_summary = pd.DataFrame({
    '항목': [
        '전체 검정 수', 'raw p < 0.05 관계 수', 'BH adjusted p < 0.05 관계 수',
        'BH 보정 후 탈락 관계 수', 'BH 통과 + 검정 가정 양호 관계 수',
        'BH 통과 + 검정 가정 주의 관계 수',
    ],
    '값': [
        len(fdr_results), int(fdr_results['raw_유의여부'].sum()), len(fdr_pass_results),
        len(dropped_after_bh), len(fdr_good_assumption_results), len(fdr_caution_results),
    ],
})
display(fdr_summary)

## 3-8. 3단계 해석

Benjamini-Hochberg 보정은 여러 검정을 동시에 수행하면서 발생할 수 있는 거짓 양성을 줄이기 위한 과정입니다. 보정 후 통과한 관계는 다음 사후분석 대상으로 삼을 수 있지만, 이것만으로 관계의 실재나 인과관계가 증명된 것은 아닙니다. 따라서 다중검정 보정 전의 통계적 관계 후보로 해석합니다.

특히 카이제곱 관계는 기대빈도 조건 문제가 크므로 보정 p-value가 작더라도 해석에 주의해야 합니다. FDR 보정은 기대빈도 부족, 소표본, 범주 불균형을 해결하지 않습니다. 효과크기도 함께 확인해 관계의 실질적 크기를 판단해야 합니다.

피해금액과 관련된 관계가 통과하더라도 어느 집단의 피해액이 더 높은지는 이 단계에서 결론 내리지 않습니다. 이는 4단계 피해금액 사후분석에서 확인합니다.

## 3-9. 3단계 최종 요약

In [ ]:
def print_relation_list(title, data):
    print(f'\n[{title}]')
    relations = data['관계'].tolist()
    if relations:
        for relation in relations:
            print('-', relation)
    else:
        print('- 없음')

print('=' * 40)
print('3단계 다중검정 보정 결과')
print('=' * 40)
print('전체 검정:', len(fdr_results))
print('raw p < 0.05:', int(fdr_results['raw_유의여부'].sum()))
print('BH adjusted p < 0.05:', len(fdr_pass_results))
print('BH 보정 후 탈락:', len(dropped_after_bh))
print('FDR 통과 + 검정 가정 양호:', len(fdr_good_assumption_results))
print('FDR 통과 + 검정 가정 주의:', len(fdr_caution_results))

print_relation_list('보정 후 관계 후보', fdr_pass_results)
print_relation_list('FDR 통과 + 검정 가정 양호', fdr_good_assumption_results)
print_relation_list('보정 후 근거 약함', fdr_weak_results)
print_relation_list('검정 가정 주의', fdr_caution_results)
print_relation_list('FDR 통과 + 효과크기 중간 이상', medium_or_larger_fdr_results)

s2_result = fdr_results.loc[fdr_results['검정ID'].eq('S2')].iloc[0]
print('\n[S2 성별 ↔ 피해금액]')
if s2_result['FDR_통과여부'] == '통과':
    print(f"{s2_result['최종판정']}: adjusted p={display_p_value(s2_result['adjusted_p_value'])}, 효과크기 해석={s2_result['효과크기_해석']}")
else:
    print('현재 데이터에서는 성별과 피해금액 차이에 대한 통계적 근거가 약했다.')

print('\n[현재 분석 미진행]')
for _, row in held_hypotheses.iterrows():
    print(f"- {row['가설ID']} {row['관계']}: {row['사유']}")

print('\n' + '=' * 40)
print('3단계 완료')
print('다음 단계: 4. 피해금액 사후분석')
print('=' * 40)

# 이후 단계에서 사용할 3단계 핵심 결과물
display(fdr_results)

# 4. 피해금액 사후분석

## 4-1. 분석 목적 및 3단계 결과 확인

3단계에서 피해금액과 관련된 S1~S4의 실제 `fdr_results`를 읽어 후속 분석 대상을 결정합니다. 이번 단계의 목적은 전체 그룹 차이가 확인된 관계에서 **어느 그룹 사이에 차이가 관찰되는지** 구체화하는 것입니다.

- 본격 사후분석 조건: BH adjusted p-value < 0.05이면서 그룹이 3개 이상
- 성별처럼 FDR 근거가 약한 관계는 기존 결과만 확인하고 검정을 반복하지 않음
- 기존 효과크기와 검정 가정 주의를 함께 해석
- 고액 피해 사례와 소표본 범주를 삭제하거나 자동 통합하지 않음

관찰된 차이는 연관성이지 인과관계를 뜻하지 않습니다.

In [ ]:
required_step4_objects = ['df_postal_analysis', 'fdr_results']
missing_step4_objects = [name for name in required_step4_objects if name not in globals()]
if missing_step4_objects:
    raise NameError(f'4단계 필수 객체가 없습니다: {missing_step4_objects}. 0~3단계를 먼저 실행해 주세요.')

step4_test_ids = ['S1', 'S2', 'S3', 'S4']
damage_fdr_results = fdr_results[fdr_results['검정ID'].isin(step4_test_ids)].copy()
damage_fdr_results = damage_fdr_results.set_index('검정ID').reindex(step4_test_ids).reset_index()

if damage_fdr_results['관계'].isna().any():
    missing_ids = damage_fdr_results.loc[damage_fdr_results['관계'].isna(), '검정ID'].tolist()
    raise ValueError(f'fdr_results에서 피해금액 검정 결과를 찾을 수 없습니다: {missing_ids}')

damage_fdr_columns = [
    '검정ID', '관계', '검정방법', 'raw_p_value', 'adjusted_p_value',
    '효과크기', '효과크기_종류', '효과크기_해석', '가정충족여부',
    'FDR_통과여부', '최종판정', '주의사항',
]
display(damage_fdr_results[damage_fdr_columns].style.format({
    'raw_p_value': display_p_value,
    'adjusted_p_value': display_p_value,
    '효과크기': '{:.4f}',
}))

step4_selected_ids = damage_fdr_results.loc[
    damage_fdr_results['adjusted_p_value'] < ALPHA, '검정ID'
].tolist()
print('4단계 본격 분석 대상:', step4_selected_ids)

## 4-2. 피해금액 사후분석 대상 선정과 공통 함수

피해금액은 강한 우측 왜도와 일부 고액 사례의 영향을 받으므로 평균과 중앙값을 함께 제시하되, 해석에서는 중앙값·사분위 범위와 전체 분포를 더 중요하게 봅니다.

Kruskal–Wallis 전체 검정은 2단계에서 이미 수행했으므로 반복하지 않습니다. 전체 검정이 FDR 보정을 통과한 3개 이상 그룹 관계에 한해 쌍별 Mann–Whitney U 검정을 하고, 한 변수 안에서 발생하는 여러 쌍의 p-value를 Benjamini–Hochberg 방식으로 다시 보정합니다. 이 방식은 별도 패키지를 설치하지 않아도 기존 SciPy와 statsmodels로 재현할 수 있습니다.

박스플롯의 확대본은 고액 사례를 삭제한 그래프가 아닙니다. 원자료 전체를 그대로 사용하고 일반적인 분포를 보기 위해 표시 축 범위만 조정합니다.

In [ ]:
from itertools import combinations

def natural_age_order(values):
    """연령대는 숫자형이면 실제 나이 순서로, 아니면 문자열 순서로 정렬합니다."""
    return sorted(values, key=lambda value: (pd.isna(value), float(value) if str(value).replace('.', '', 1).isdigit() else str(value)))

def make_deep_amount_summary(data, group_column, order=None):
    summary = (
        data.dropna(subset=[group_column, '피해액'])
        .groupby(group_column, observed=False)['피해액']
        .agg(
            표본수='count', 평균_피해액='mean', 중앙값_피해액='median',
            최소값='min', Q1=lambda values: values.quantile(0.25),
            Q3=lambda values: values.quantile(0.75), 최대값='max',
        )
    )
    summary['IQR'] = summary['Q3'] - summary['Q1']
    if order is not None:
        existing_order = [value for value in order if value in summary.index]
        summary = summary.reindex(existing_order)
    return summary

def median_descending_order(data, group_column):
    return (
        data.dropna(subset=[group_column, '피해액'])
        .groupby(group_column, observed=False)['피해액']
        .median().sort_values(ascending=False).index.tolist()
    )

def show_damage_boxplots(data, group_column, order, title, figure_height=6, horizontal=True):
    plot_data = data.dropna(subset=[group_column, '피해액']).copy()
    labels = [str(value) for value in order]
    values = [
        plot_data.loc[plot_data[group_column].eq(value), '피해액'].to_numpy() / WON_PER_MANWON
        for value in order
    ]

    fig, ax = plt.subplots(figsize=(11, figure_height))
    if horizontal:
        ax.boxplot(values, labels=labels, vert=False, showfliers=True)
        ax.set_xlabel('피해금액(만원)')
        ax.set_ylabel(group_column)
        ax.grid(axis='x', alpha=0.25)
    else:
        ax.boxplot(values, labels=labels, vert=True, showfliers=True)
        ax.set_xlabel(group_column)
        ax.set_ylabel('피해금액(만원)')
        ax.grid(axis='y', alpha=0.25)
    ax.set_title(f'{title} - 전체 범위')
    plt.tight_layout()
    plt.show()

    all_amounts = plot_data['피해액'] / WON_PER_MANWON
    q1, q3 = all_amounts.quantile([0.25, 0.75])
    upper_fence = q3 + 1.5 * (q3 - q1)
    needs_zoom = bool(all_amounts.max() > upper_fence * 1.5 and upper_fence > 0)
    if needs_zoom:
        fig, ax = plt.subplots(figsize=(11, figure_height))
        if horizontal:
            ax.boxplot(values, labels=labels, vert=False, showfliers=True)
            ax.set_xlim(left=0, right=upper_fence)
            ax.set_xlabel('피해금액(만원)')
            ax.set_ylabel(group_column)
            ax.grid(axis='x', alpha=0.25)
        else:
            ax.boxplot(values, labels=labels, vert=True, showfliers=True)
            ax.set_ylim(bottom=0, top=upper_fence)
            ax.set_xlabel(group_column)
            ax.set_ylabel('피해금액(만원)')
            ax.grid(axis='y', alpha=0.25)
        ax.set_title(f'{title} - 일반 분포 확대(IQR 상한까지)')
        plt.tight_layout()
        plt.show()
        print('확대본은 전체 데이터를 유지하고 피해금액 축의 표시 범위만 조정했습니다.')
    return needs_zoom

def run_pairwise_mannwhitney_bh(data, group_column, analysis_variable, order):
    """전체 검정 통과 뒤 어떤 그룹 쌍이 다른지 확인하고 쌍별 거짓 양성을 제어합니다."""
    rows = []
    clean = data.dropna(subset=[group_column, '피해액'])
    medians = clean.groupby(group_column, observed=False)['피해액'].median()
    counts = clean.groupby(group_column, observed=False)['피해액'].count()

    for group_1, group_2 in combinations(order, 2):
        values_1 = clean.loc[clean[group_column].eq(group_1), '피해액'].to_numpy()
        values_2 = clean.loc[clean[group_column].eq(group_2), '피해액'].to_numpy()
        statistic, raw_p_value = mannwhitneyu(
            values_1, values_2, alternative='two-sided', method='asymptotic'
        )
        caution = []
        if min(len(values_1), len(values_2)) < 5:
            caution.append('한쪽 이상 n<5로 해석 주의')
        if len(values_1) == 1 or len(values_2) == 1:
            caution.append('단일 관측 그룹 포함')
        rows.append({
            '분석변수': analysis_variable, '그룹1': group_1, '그룹2': group_2,
            'raw_p_value': float(raw_p_value),
            '그룹1_중앙값': float(medians[group_1]), '그룹2_중앙값': float(medians[group_2]),
            '그룹1_n': int(counts[group_1]), '그룹2_n': int(counts[group_2]),
            '주의사항': '; '.join(caution) if caution else '특이 주의 없음',
        })

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    reject, adjusted, _, _ = multipletests(
        result['raw_p_value'].to_numpy(), alpha=ALPHA, method='fdr_bh'
    )
    result['adjusted_p_value'] = adjusted.astype(float)
    result['보정후_유의'] = np.where(reject, '유의', '유의하지 않음')
    return result[[
        '분석변수', '그룹1', '그룹2', 'raw_p_value', 'adjusted_p_value', '보정후_유의',
        '그룹1_중앙값', '그룹2_중앙값', '그룹1_n', '그룹2_n', '주의사항',
    ]]

def get_fdr_row(test_id):
    return damage_fdr_results.loc[damage_fdr_results['검정ID'].eq(test_id)].iloc[0]

def print_existing_test_result(test_id):
    row = get_fdr_row(test_id)
    print(
        f"{test_id} {row['관계']}: BH adjusted p={display_p_value(row['adjusted_p_value'])}, "
        f"효과크기={row['효과크기']:.4f}({row['효과크기_해석']}), 최종판정={row['최종판정']}"
    )

def conditional_posthoc(test_id, data, group_column, analysis_variable, order):
    fdr_row = get_fdr_row(test_id)
    if fdr_row['adjusted_p_value'] >= ALPHA:
        print('전체 그룹 차이가 FDR 보정을 통과하지 않아 사후검정을 수행하지 않습니다.')
        return pd.DataFrame()
    if len(order) < 3:
        print('그룹이 3개 미만이므로 다그룹 사후검정을 수행하지 않습니다.')
        return pd.DataFrame()
    result = run_pairwise_mannwhitney_bh(data, group_column, analysis_variable, order)
    display(result.style.format({
        'raw_p_value': display_p_value,
        'adjusted_p_value': display_p_value,
        '그룹1_중앙값': '{:,.0f}',
        '그룹2_중앙값': '{:,.0f}',
    }))
    print('보정 후 유의한 pair 수:', int(result['보정후_유의'].eq('유의').sum()))
    return result

posthoc_tables = {}
zoom_plot_usage = {}

## 4-3. 연령대 ↔ 피해금액

### 그룹별 요약표

연령대별 표본 수, 평균, 중앙값, 사분위수와 전체 범위를 먼저 확인합니다. 연령대는 실제 데이터의 자연스러운 나이 순서로 정렬합니다.

In [ ]:
age_order = natural_age_order(df_postal_analysis['연령대'].dropna().unique().tolist())
age_damage_summary = make_deep_amount_summary(df_postal_analysis, '연령대', age_order)
display(age_damage_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}', '최소값': '{:,.0f}',
    'Q1': '{:,.0f}', 'Q3': '{:,.0f}', '최대값': '{:,.0f}', 'IQR': '{:,.0f}',
}))

### 박스플롯

전체 범위 그래프는 초고액 피해를 포함한 원자료 분포를 보여줍니다. 고액 사례 때문에 상자 부분이 지나치게 압축되는 경우에만 동일한 원자료로 축 확대본을 자동 추가합니다.

In [ ]:
zoom_plot_usage['연령대'] = show_damage_boxplots(
    df_postal_analysis, '연령대', age_order, '연령대별 피해금액', figure_height=5.5, horizontal=False
)

### 전체 검정 결과 연결 및 필요한 사후검정

2단계 Kruskal–Wallis 검정을 다시 실행하지 않고 3단계 결과를 연결합니다. FDR 통과 조건을 만족할 때만 연령대 쌍별 비교를 수행합니다.

In [ ]:
print_existing_test_result('S1')
age_posthoc_results = conditional_posthoc(
    'S1', df_postal_analysis, '연령대', '연령대', age_order
)
posthoc_tables['연령대'] = age_posthoc_results

In [ ]:
age_low_group = age_damage_summary['중앙값_피해액'].idxmin()
age_high_group = age_damage_summary['중앙값_피해액'].idxmax()
age_significant_pairs = (
    age_posthoc_results[age_posthoc_results['보정후_유의'].eq('유의')]
    if not age_posthoc_results.empty else age_posthoc_results
)
print('[연령대 결과 해석]')
print(f'- 중앙값이 가장 낮은 연령대: {age_low_group}')
print(f'- 중앙값이 가장 높은 연령대: {age_high_group}')
print(f'- BH 보정 후 유의한 연령대 pair: {len(age_significant_pairs)}개')
print('- 연령대별 피해금액 분포의 차이를 관찰한 결과이며, 연령이 피해금액의 원인이라는 뜻은 아닙니다.')
print('- 연령대는 피해금액 분포 차이를 설명하는 보조 특성이며 단독 차단 기준이나 위험 원인으로 사용하지 않습니다.')

## 4-4. 성별 ↔ 피해금액

3단계에서 성별 관계의 FDR 근거가 약하면 새로운 검정이나 과도한 그래프를 추가하지 않습니다. 비교 가능한 최소 요약과 기존 판정만 확인합니다.

In [ ]:
gender_order = sorted(df_postal_analysis['피해자 성별'].dropna().unique().tolist())
gender_damage_summary = make_deep_amount_summary(df_postal_analysis, '피해자 성별', gender_order)
display(gender_damage_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}', '최소값': '{:,.0f}',
    'Q1': '{:,.0f}', 'Q3': '{:,.0f}', '최대값': '{:,.0f}', 'IQR': '{:,.0f}',
}))
print_existing_test_result('S2')
gender_fdr_row = get_fdr_row('S2')
if gender_fdr_row['adjusted_p_value'] >= ALPHA:
    print('현재 데이터에서는 성별에 따른 피해금액 차이의 통계적 근거가 약하므로 4단계 사후분석 대상에서 제외합니다.')
else:
    print('성별은 2개 그룹이므로 기존 Mann-Whitney U 결과로 판단하며 다그룹 사후검정은 수행하지 않습니다.')

## 4-5. 사기유형 ↔ 피해금액

### 그룹별 요약표와 박스플롯

사기유형은 중앙 피해금액이 큰 순서로 정렬합니다. 표본이 매우 적은 범주도 삭제하거나 자동 통합하지 않고 표와 사후검정 주의사항에 그대로 기록합니다.

In [ ]:
fraud_type_order = median_descending_order(df_postal_analysis, '사기유형')
fraud_type_damage_summary = make_deep_amount_summary(
    df_postal_analysis, '사기유형', fraud_type_order
)
display(fraud_type_damage_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}', '최소값': '{:,.0f}',
    'Q1': '{:,.0f}', 'Q3': '{:,.0f}', '최대값': '{:,.0f}', 'IQR': '{:,.0f}',
}))
print('정렬 기준: 중앙 피해금액 내림차순')

In [ ]:
fraud_figure_height = max(5.5, 0.55 * len(fraud_type_order) + 2)
zoom_plot_usage['사기유형'] = show_damage_boxplots(
    df_postal_analysis, '사기유형', fraud_type_order,
    '사기유형별 피해금액', figure_height=fraud_figure_height
)

### 전체 검정 결과 연결 및 필요한 사후검정

In [ ]:
print_existing_test_result('S3')
fraud_type_posthoc_results = conditional_posthoc(
    'S3', df_postal_analysis, '사기유형', '사기유형', fraud_type_order
)
posthoc_tables['사기유형'] = fraud_type_posthoc_results

In [ ]:
fraud_low_group = fraud_type_damage_summary['중앙값_피해액'].idxmin()
fraud_high_group = fraud_type_damage_summary['중앙값_피해액'].idxmax()
fraud_significant_pairs = (
    fraud_type_posthoc_results[fraud_type_posthoc_results['보정후_유의'].eq('유의')]
    if not fraud_type_posthoc_results.empty else fraud_type_posthoc_results
)
fraud_small_groups = fraud_type_damage_summary[fraud_type_damage_summary['표본수'] < 5].index.tolist()
print('[사기유형 결과 해석]')
print(f'- 중앙값이 가장 낮은 유형: {fraud_low_group}')
print(f'- 중앙값이 가장 높은 유형: {fraud_high_group}')
print(f'- BH 보정 후 유의한 사기유형 pair: {len(fraud_significant_pairs)}개')
print(f'- 표본 5건 미만 유형: {fraud_small_groups if fraud_small_groups else "없음"}')
print('- 특정 사기유형에서 상대적으로 높은 피해금액 분포가 관찰될 수 있으나 인과관계로 해석하지 않습니다.')

## 4-6. 사칭기관 ↔ 피해금액

### 그룹별 요약표와 박스플롯

사칭기관은 중앙 피해금액이 큰 순서로 정렬합니다. 소표본 기관도 원자료의 일부이므로 삭제·통합하지 않습니다.

In [ ]:
institution_order = median_descending_order(df_postal_analysis, '사칭기관')
institution_damage_summary = make_deep_amount_summary(
    df_postal_analysis, '사칭기관', institution_order
)
display(institution_damage_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}', '최소값': '{:,.0f}',
    'Q1': '{:,.0f}', 'Q3': '{:,.0f}', '최대값': '{:,.0f}', 'IQR': '{:,.0f}',
}))
print('정렬 기준: 중앙 피해금액 내림차순')

In [ ]:
institution_figure_height = max(6, 0.48 * len(institution_order) + 2)
zoom_plot_usage['사칭기관'] = show_damage_boxplots(
    df_postal_analysis, '사칭기관', institution_order,
    '사칭기관별 피해금액', figure_height=institution_figure_height
)

### 전체 검정 결과 연결 및 필요한 사후검정

In [ ]:
print_existing_test_result('S4')
institution_posthoc_results = conditional_posthoc(
    'S4', df_postal_analysis, '사칭기관', '사칭기관', institution_order
)
posthoc_tables['사칭기관'] = institution_posthoc_results

In [ ]:
institution_low_group = institution_damage_summary['중앙값_피해액'].idxmin()
institution_high_group = institution_damage_summary['중앙값_피해액'].idxmax()
institution_significant_pairs = (
    institution_posthoc_results[institution_posthoc_results['보정후_유의'].eq('유의')]
    if not institution_posthoc_results.empty else institution_posthoc_results
)
institution_small_groups = institution_damage_summary[
    institution_damage_summary['표본수'] < 5
].index.tolist()
print('[사칭기관 결과 해석]')
print(f'- 중앙값이 가장 낮은 기관: {institution_low_group}')
print(f'- 중앙값이 가장 높은 기관: {institution_high_group}')
print(f'- BH 보정 후 유의한 사칭기관 pair: {len(institution_significant_pairs)}개')
print(f'- 표본 5건 미만 기관: {institution_small_groups if institution_small_groups else "없음"}')
print('- 사칭기관별 피해금액 분포는 실제 피해사례 구조를 설명하는 참고 정보이며 단독 차단 기준으로 사용하지 않습니다.')

## 4-7. 사후검정 결과 통합

사후검정을 실제로 수행한 관계의 결과만 하나의 표로 합칩니다. adjusted p-value는 각 분석변수 안의 모든 pair에 BH 보정을 적용한 값입니다.

In [ ]:
non_empty_posthoc_tables = [table for table in posthoc_tables.values() if not table.empty]
all_posthoc_results = (
    pd.concat(non_empty_posthoc_tables, ignore_index=True)
    if non_empty_posthoc_tables
    else pd.DataFrame(columns=[
        '분석변수', '그룹1', '그룹2', 'raw_p_value', 'adjusted_p_value', '보정후_유의',
        '그룹1_중앙값', '그룹2_중앙값', '그룹1_n', '그룹2_n', '주의사항',
    ])
)
display(all_posthoc_results.style.format({
    'raw_p_value': display_p_value,
    'adjusted_p_value': display_p_value,
    '그룹1_중앙값': '{:,.0f}',
    '그룹2_중앙값': '{:,.0f}',
}))
print('전체 사후비교 수:', len(all_posthoc_results))
print('BH 보정 후 유의한 pair 수:', int(all_posthoc_results['보정후_유의'].eq('유의').sum()))

## 4-8. 피해금액 사후분석 결과표

보정 p-value, 2단계 효과크기, 그룹별 중앙값과 사후검정 결과를 함께 봅니다. p-value만으로 관계의 실질적 중요도나 서비스 적용 여부를 확정하지 않습니다.

In [ ]:
summary_by_test = {
    'S1': age_damage_summary,
    'S2': gender_damage_summary,
    'S3': fraud_type_damage_summary,
    'S4': institution_damage_summary,
}
posthoc_by_test = {
    'S1': age_posthoc_results,
    'S2': pd.DataFrame(),
    'S3': fraud_type_posthoc_results,
    'S4': institution_posthoc_results,
}

deep_rows = []
for test_id in step4_test_ids:
    fdr_row = get_fdr_row(test_id)
    summary = summary_by_test[test_id]
    posthoc = posthoc_by_test[test_id]
    significant = (
        posthoc[posthoc['보정후_유의'].eq('유의')] if not posthoc.empty else posthoc
    )
    cautions = []
    small_groups = summary[summary['표본수'] < 5].index.astype(str).tolist()
    if small_groups:
        cautions.append(f'표본 5건 미만: {small_groups}')
    if '주의' in str(fdr_row['가정충족여부']):
        cautions.append(str(fdr_row['가정충족여부']))
    if fdr_row['adjusted_p_value'] >= ALPHA:
        cautions.append('FDR 보정 후 근거 약함')

    deep_rows.append({
        '검정ID': test_id,
        '관계': fdr_row['관계'],
        'BH_adjusted_p_value': float(fdr_row['adjusted_p_value']),
        '효과크기': float(fdr_row['효과크기']),
        '효과크기_종류': fdr_row['효과크기_종류'],
        '효과크기_해석': fdr_row['효과크기_해석'],
        '가장_낮은_중앙값_그룹': summary['중앙값_피해액'].idxmin(),
        '가장_높은_중앙값_그룹': summary['중앙값_피해액'].idxmax(),
        '사후검정_수행여부': '수행' if not posthoc.empty else '미수행',
        '유의한_pair_수': int(len(significant)),
        '주의사항': '; '.join(cautions) if cautions else '특이 주의 없음',
    })

damage_deep_results = pd.DataFrame(deep_rows)
display(damage_deep_results.style.format({
    'BH_adjusted_p_value': display_p_value,
    '효과크기': '{:.4f}',
}))

## 4-9. 서비스 연결 시 참고사항

우체국 분석 결과는 서비스 위험 규칙을 직접 만드는 데이터가 아닙니다. 실제 피해사례의 특성을 이해하고 이후 통화 기반 위험 탐지 결과를 해석하는 참고 근거로 사용합니다.

In [ ]:
service_interpretation_notes = pd.DataFrame([
    {
        '변수': '연령대',
        '해석 참고': '피해금액 분포 차이를 설명하는 보조 특성',
        '적용 원칙': '연령 단독 차단 기준으로 사용하지 않으며 특정 연령을 위험의 원인으로 해석하지 않음',
    },
    {
        '변수': '사기유형',
        '해석 참고': '유형별 피해규모 차이와 통화 분석의 사기 시나리오를 해석하는 보조 근거',
        '적용 원칙': '우체국 결과만으로 위험 가중치를 직접 확정하지 않음',
    },
    {
        '변수': '사칭기관',
        '해석 참고': '실제 피해사례의 사칭 구조 설명 및 LLM 구조화·RAG 사례 설명 참고',
        '적용 원칙': '단독 차단 기준이나 확정 위험점수로 사용하지 않음',
    },
])
display(service_interpretation_notes)

## 4-10. 4단계 최종 요약

아래 출력은 저장된 숫자를 하드코딩하지 않고 현재 실행 결과에서 자동 생성합니다. 통계적 차이, 효과크기, 실제 분포와 소표본 문제를 함께 확인해야 합니다.

In [ ]:
def format_pair_list(data):
    if data.empty:
        return '없음'
    significant = data[data['보정후_유의'].eq('유의')]
    if significant.empty:
        return '없음'
    return ', '.join(
        f"{row['그룹1']} ↔ {row['그룹2']}"
        for _, row in significant.iterrows()
    )

final_config = [
    ('S1', '연령대', age_damage_summary, age_posthoc_results),
    ('S2', '성별', gender_damage_summary, pd.DataFrame()),
    ('S3', '사기유형', fraud_type_damage_summary, fraud_type_posthoc_results),
    ('S4', '사칭기관', institution_damage_summary, institution_posthoc_results),
]

print('=' * 40)
print('4단계 피해금액 사후분석 결과')
print('=' * 40)
for test_id, label, summary, posthoc in final_config:
    row = get_fdr_row(test_id)
    small_groups = summary[summary['표본수'] < 5].index.astype(str).tolist()
    cautions = []
    if small_groups:
        cautions.append(f'표본 5건 미만: {small_groups}')
    if row['adjusted_p_value'] >= ALPHA:
        cautions.append('현재 데이터에서는 FDR 보정 후 근거 약함')
    if '주의' in str(row['가정충족여부']):
        cautions.append(str(row['가정충족여부']))

    print(f'\n[{label} ↔ 피해금액]')
    print('BH adjusted p-value:', display_p_value(row['adjusted_p_value']))
    print(f"효과크기: {row['효과크기']:.4f} ({row['효과크기_종류']}, {row['효과크기_해석']})")
    print('가장 낮은 중앙 피해 그룹:', summary['중앙값_피해액'].idxmin())
    print('가장 높은 중앙 피해 그룹:', summary['중앙값_피해액'].idxmax())
    print('유의한 사후 pair:', format_pair_list(posthoc))
    print('주의사항:', '; '.join(cautions) if cautions else '특이 주의 없음')

print('\n[서비스 해석 참고]')
for _, candidate in service_interpretation_notes.iterrows():
    print(f"- {candidate['변수']}: {candidate['해석 참고']} ({candidate['적용 원칙']})")

print('\n[그래프 확인]')
print('필수 전체 범위 박스플롯: 연령대, 사기유형, 사칭기관')
print('축 확대본 사용 여부:', zoom_plot_usage)
print('확대본도 원자료를 삭제하지 않고 축 범위만 조정했습니다.')

print('\n' + '=' * 40)
print('4단계 완료')
print('현재 범위에서 우체국 통계분석 종료')
print('=' * 40)

# 우체국 데이터 분석 최종 정리

---

## 분석 목적

우체국 데이터는 경찰청 집계통계보다 개별 피해사례에 가까운 데이터입니다. 주요 변수는 연령대, 성별, 피해금액, 사기유형, 사칭기관입니다.

> 핵심 질문: 실제 전화 보이스피싱 피해사례에서는 어떤 피해자 특성과 피해 특성이 나타나는가?

이 데이터는 피해자 구성과 피해금액 분포를 파악하고, 사기유형·사칭기관별 피해 특성과 실제 피해금액 간 통계적 관계를 확인하는 데 사용합니다. 우체국 분석 결과는 서비스 위험 규칙을 직접 만드는 데이터가 아니라 실제 피해사례의 특성을 이해하고 이후 통화 기반 위험 탐지 결과를 해석하는 참고 근거입니다.

## 분석 완료 범위

- 0단계 데이터 준비 완료
- 1단계 기본 EDA 완료
- 2단계 전체 관계 스크리닝 완료
- 3단계 Benjamini-Hochberg 다중검정 보정 완료
- 4단계 피해금액 사후분석 완료

> 현재 우체국 데이터 분석은 4단계까지 수행한 결과로 충분한 EDA 및 통계적 인사이트를 확보했으므로 여기서 종료합니다.

## 0~4단계 최종 결과 요약

| 단계 | 분석 내용 | 핵심 결과 | 최종 활용 |
| --- | --- | --- | --- |
| 0단계 | 데이터 준비 | 원본 184건에서 전화 보이스피싱을 선택하고 투자사기·중복후보를 제외하여 최종 106건을 분석 표본으로 구성 | 분석 기반 |
| 1단계 | 기본 EDA | 평균 45,574,686원, 중앙값 9,000,000원, Q1 500,000원, Q3 39,750,000원으로 오른쪽 꼬리가 긴 분포 확인 | 피해자·피해규모·유형·기관 대시보드 후보 |
| 2단계 | 관계 스크리닝 | S1~S9 총 9개 검정 수행. 성별 ↔ 피해금액을 제외한 8개 관계의 raw p-value가 0.05 미만 | 통계 관계 후보 |
| 3단계 | FDR 보정 | 9개 중 8개가 BH 보정 통과. 피해금액 관계 S1·S3·S4는 효과크기가 컸고, 범주형 관계 S5~S9는 모두 기대빈도 조건 주의 | 최종 판단 근거 |
| 4단계 | 피해금액 사후분석 | 연령대 7개, 사기유형 2개, 사칭기관 7개 pair가 각 분석 내 BH 보정 후 유의. 성별은 전체 검정 근거가 약해 사후검정 미수행 | 핵심 인사이트와 해석 참고 |

## 핵심 관계 최종 요약

| 관계 | 분석 결과 | 효과크기 | 검정 가정 | 해석 |
| --- | --- | --- | --- | --- |
| 연령대 ↔ 피해금액 (S1) | BH 통과, adjusted p=3.537e-07 | ε²=0.3489, 큼 | 대체로 충족 | 연령대별 피해금액 분포 차이가 확인되었으나 연령이 원인이라는 의미는 아님 |
| 성별 ↔ 피해금액 (S2) | BH 미통과, adjusted p=0.8190 | rank-biserial 절댓값=0.0262, 매우 작음 | 대체로 충족 | 현재 표본에서는 성별에 따른 피해금액 차이의 통계적 근거가 약함 |
| 사기유형 ↔ 피해금액 (S3) | BH 통과, adjusted p=1.617e-05 | ε²=0.2569, 큼 | 소표본 주의 | 유형별 분포 차이가 있으나 1~2건 범주를 포함하므로 작은 범주를 과대해석하지 않음 |
| 사칭기관 ↔ 피해금액 (S4) | BH 통과, adjusted p=1.922e-06 | ε²=0.3075, 큼 | 소표본 주의 | 기관별 분포 차이가 있으나 2~4건 범주를 포함하므로 확정적 순위로 해석하지 않음 |
| 사기유형 ↔ 사칭기관 (S5) | BH 통과, adjusted p=2.796e-27 | Cramér's V=0.6062, 큼 | 주의: 기대빈도 5 미만 31/36셀 | 통계적 관계 후보는 나타났으나 검정 가정에 주의가 필요함 |
| 연령대 ↔ 사기유형 (S6) | BH 통과, adjusted p=7.596e-14 | Cramér's V=0.4772, 큼 | 주의: 기대빈도 5 미만 30/36셀 | 탐색적 관계 후보이며 연령을 위험 원인으로 해석하지 않음 |
| 연령대 ↔ 사칭기관 (S7) | BH 통과, adjusted p=1.084e-11 | Cramér's V=0.4499, 큼 | 주의: 기대빈도 5 미만 30/36셀 | 탐색적 관계 후보이며 범주 불균형을 함께 고려함 |
| 성별 ↔ 사기유형 (S8) | BH 통과, adjusted p=0.0017 | Cramér's V=0.4297, 중간 | 주의: 기대빈도 5 미만 8/12셀 | p-value가 작아도 기대빈도 조건 때문에 확정적 관계로 해석하지 않음 |
| 성별 ↔ 사칭기관 (S9) | BH 통과, adjusted p=0.0007 | Cramér's V=0.4539, 중간 | 주의: 기대빈도 5 미만 8/12셀 | 통계적 관계 후보는 나타났으나 검정 가정에 주의가 필요함 |

### 피해금액 사후분석에서 확인한 집단 차이

- 연령대: 20↔30, 20↔40, 20↔60, 20↔70, 30↔40, 40↔60, 40↔70의 7개 pair가 보정 후 유의했습니다. 중앙값은 70대 50,500,000원, 40대 490,000원으로 가장 높고 낮았지만 인과관계나 위험 순위로 해석하지 않습니다.
- 사기유형: 카드배송사칭↔가족납치·상해 협박, 사건연루조사↔가족납치·상해 협박의 2개 pair가 보정 후 유의했습니다. `기타` 등 단일 관측 범주는 중앙값 순위 해석에서 제외해 보아야 합니다.
- 사칭기관: 금감원·금융위↔경찰·검찰·법원/개인/기타, 할부금융↔개인/기타, 경찰·검찰·법원↔개인/기타의 7개 pair가 보정 후 유의했습니다. 할부금융은 4건이므로 관련 pair 해석에 특히 주의합니다.

## 피해금액 해석 시 주의사항

- 피해금액은 오른쪽 꼬리가 긴 분포이며 일부 고액 사례가 평균을 크게 끌어올릴 수 있습니다.
- 평균뿐 아니라 중앙값, Q1, Q3와 전체 분포를 함께 봅니다.
- IQR 기준 이상치 후보는 오류 데이터라는 의미가 아니며 실제 고액 피해사례이므로 임의로 삭제하지 않습니다.
- 집단 비교에서는 단순 평균 순위보다 분포와 중앙값을 함께 해석합니다.
- 통계적으로 유의한 차이는 인과관계를 의미하지 않습니다.
- 작은 표본의 범주는 과도하게 해석하지 않습니다.

## 고액피해 기준에 대한 최종 판단

현재 프로젝트에서는 Q3, 상위 25%, 고정 금액, 평균+표준편차 등의 기준으로 별도의 `고액피해 여부` Target을 핵심 분석에 추가하지 않습니다.

1. 기준 선택에 따라 고액/비고액 분류가 달라질 수 있습니다.
2. 피해금액을 이진화하면 실제 금액 크기 정보가 손실됩니다.
3. 현재 서비스는 고액피해자를 예측하는 서비스가 아닙니다.
4. 이미 실제 피해금액 자체를 이용한 관계 분석을 수행했습니다.

향후 대시보드에서 피해 심각성을 설명하는 보조 KPI로 특정 금액 이상 피해 사례 비중을 사용하는 것은 별도 검토할 수 있습니다. 이번 노트북에서는 추가 고액피해 가설검정을 진행하지 않습니다.

## 현재 우체국 데이터에서 진행하지 않는 추가 분석

- 별도의 피해자 ↔ 사기수법 추가 심층검정
- 고액피해 여부 기반 U3·U7·U8 핵심 검정
- 복합 관계 분석, 회귀분석, 군집분석, 우체국 데이터 기반 ML
- 경찰청 데이터와의 억지 결합
- 정상통화 vs 보이스피싱 비교, 심리압박 분석, 통화 유사도 분석

> 정상통화 비교, 심리압박, 유사도, ML은 우체국 데이터가 아니라 별도의 통화·전사 데이터 분석 영역에서 진행합니다.

## 대시보드 활용 검토 후보

아래 표는 후보 목록이며 O/△/X, 최우선, 최종 제외 판정은 이후 별도 선별 단계에서 결정합니다.

| 분석 | 대시보드에서 보여줄 수 있는 내용 | 시각화 후보 |
| --- | --- | --- |
| 분석 표본 | 실제 분석에 사용한 피해사례 범위 | KPI / 설명 |
| 연령대 분포 | 피해사례의 연령 구성 | 막대그래프 |
| 성별 분포 | 피해사례의 성별 구성 | 막대그래프 |
| 피해금액 분포 | 피해규모의 편향과 고액 사례 존재 | 히스토그램 / KPI |
| 피해금액 전체 분포 | 중앙값·IQR·고액 사례 | 박스플롯 |
| 사기유형 분포 | 피해사례의 사기유형 구성 | 막대그래프 |
| 사칭기관 분포 | 피해사례의 사칭대상 구성 | 가로 막대그래프 |
| 연령대 ↔ 피해금액 | 연령대별 피해금액 분포 | 박스플롯 |
| 성별 ↔ 피해금액 | 성별 피해금액 비교 | 박스플롯 또는 제외 검토 |
| 사기유형 ↔ 피해금액 | 유형별 피해금액 분포 | 박스플롯 |
| 사칭기관 ↔ 피해금액 | 사칭기관별 피해금액 분포 | 박스플롯 |
| 사기유형 ↔ 사칭기관 | 반복되는 시나리오 조합 | 히트맵 후보 |

## 우체국 데이터 분석 완료

- 0단계 데이터 준비 완료
- 1단계 기본 EDA 완료
- 2단계 전체 관계 스크리닝 완료
- 3단계 다중검정 보정 완료
- 4단계 피해금액 사후분석 완료
- 현재 범위에서 우체국 통계분석 종료
- 고액피해 여부 기반 추가 검정은 진행하지 않음
- 우체국 데이터 기반 별도 ML은 진행하지 않음
- 이후 작업은 대시보드용 결과 선별 및 시각화

> 우체국 데이터는 실제 피해사례의 특성과 피해금액 관계를 설명하는 데이터로 활용하며, 현재 서비스의 실시간 보이스피싱 탐지 모델을 직접 학습시키는 데이터로 사용하지 않습니다.